pull data

In [30]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import pyodbc
# -----------------------------------------------------------------------------
# ANALYSIS PERIOD
# Summer 2026
# -----------------------------------------------------------------------------

analysis_start_date = pd.Timestamp("2026-03-29")
analysis_end_date = pd.Timestamp("2026-10-24")

arrival_dates = pd.date_range(
    analysis_start_date,
    analysis_end_date,
    freq="D"
)

print(f"Analysis period:")
print(f"Start: {analysis_start_date.date()}")
print(f"End:   {analysis_end_date.date()}")

# -----------------------------------------------------------------------------
# FORECAST FILE LOCATION
# -----------------------------------------------------------------------------

user = 'jamie_douglas'

dsn = 'AzureConnection'
# user = 'christina_k'

username = f'{user}@edinburghairport.com'

connect = pyodbc.connect('DSN=' + dsn + ';')

forecast_folder = Path(
    rf"C:\Users\{user}\Edinburgh Airport Limited\Business Planning - Documents\02. Capacity Areas\09. Car Parking\01. FastPark\01 Short-Term Models\Outputs\Daily Forecast Output"
)

# -----------------------------------------------------------------------------
# LOAD ACTUALS
# -----------------------------------------------------------------------------

# Assumes:
# - connect already exists
# - pyodbc/sqlalchemy connection already created

# We need actuals for the last completed week only

sql = f"""
SELECT *
FROM FastPark.v_EntryAndExits
WHERE [CheckInStarted] >= '{analysis_start_date.strftime('%Y-%m-%d')}'
    AND [CheckInStarted] < '{(analysis_end_date + pd.Timedelta(days=1)).strftime('%Y-%m-%d')}'
"""

fastpark_actual_entries_df = pd.read_sql(sql, connect)

fastpark_actual_entries_df = fastpark_actual_entries_df.drop_duplicates(subset='BookingReference')


sql = f"""
SELECT *
FROM FastPark.v_EntryAndExits
WHERE [ActualCheckedOutDate] >= '{analysis_start_date.strftime('%Y-%m-%d')}'
    AND [ActualCheckedOutDate] < '{(analysis_end_date + pd.Timedelta(days=1)).strftime('%Y-%m-%d')}'
"""

fastpark_actual_exits_df = pd.read_sql(sql, connect)

fastpark_actual_exits_df = fastpark_actual_exits_df.drop_duplicates(subset='BookingReference')

# -----------------------------------------------------------------------------
# PREP ACTUALS
# -----------------------------------------------------------------------------

# Entries
fastpark_actual_entries_df["Date"] = pd.to_datetime(
    fastpark_actual_entries_df["CheckInStarted"]
).dt.date

actual_entries_daily = (
    fastpark_actual_entries_df
    .groupby("Date")
    .size()
    .reset_index(name="ActualEntries")
)

# Exits
fastpark_actual_exits_df["Date"] = pd.to_datetime(
    fastpark_actual_exits_df["ActualCheckedOutDate"]
).dt.date

actual_exits_daily = (
    fastpark_actual_exits_df
    .groupby("Date")
    .size()
    .reset_index(name="ActualExits")
)

# Merge actuals
actuals_daily = pd.merge(
    actual_entries_daily,
    actual_exits_daily,
    on="Date",
    how="outer"
).fillna(0)

actuals_daily["Date"] = pd.to_datetime(actuals_daily["Date"])

print("\nActuals daily:")
print(actuals_daily.head())

# -----------------------------------------------------------------------------
# LEAD TIMES TO CHECK
# -----------------------------------------------------------------------------

lead_times = [42, 35, 28, 21, 14, 7]

# Results dictionary
entries_accuracy_results = {}
exits_accuracy_results = {}
# Results dictionary
entries_accuracy_results = {}
exits_accuracy_results = {}

# NEW
entries_forecast_results = {}
exits_forecast_results = {}

entries_actual_results = {}
exits_actual_results = {}
# -----------------------------------------------------------------------------
# LOOP THROUGH EACH DAY IN LAST WEEK
# -----------------------------------------------------------------------------

for target_date in arrival_dates:

    target_date_only = target_date.date()

    # Get actuals
    actual_row = actuals_daily[
        actuals_daily["Date"].dt.date == target_date_only
    ]

    if actual_row.empty:
        print(f"\nNo actuals found for {target_date_only}")
        continue

    actual_entries = actual_row["ActualEntries"].iloc[0]
    actual_exits = actual_row["ActualExits"].iloc[0]

    entries_accuracy_results[target_date_only] = {}
    exits_accuracy_results[target_date_only] = {}
    
    entries_forecast_results[target_date_only] = {}
    exits_forecast_results[target_date_only] = {}

    entries_actual_results[target_date_only] = {}
    exits_actual_results[target_date_only] = {}
    # -------------------------------------------------------------------------
    # CHECK EACH LEAD TIME
    # -------------------------------------------------------------------------

    for lead in lead_times:

        # Forecast file date
        forecast_run_date = target_date - pd.Timedelta(days=lead)

        if forecast_run_date.strftime('%Y-%m-%d') == '2026-06-22':
            forecast_filename = (
                f"{forecast_run_date.strftime('%Y-%m-%d')}_daily_forecast - Copy.csv"
            )
        else:
            forecast_filename = (
                f"{forecast_run_date.strftime('%Y-%m-%d')}_daily_forecast.csv"
            )

        forecast_path = forecast_folder / forecast_filename

        print(f"\nChecking forecast file:")
        print(f"Target date: {target_date_only}")
        print(f"Lead time: {lead}")
        print(f"Forecast file: {forecast_filename}")

        # ---------------------------------------------------------------------
        # FILE EXISTS?
        # ---------------------------------------------------------------------

        if not forecast_path.exists():
            print("Forecast file NOT found - skipping")
            entries_accuracy_results[target_date_only][lead] = np.nan
            exits_accuracy_results[target_date_only][lead] = np.nan
            continue

        try:

            # -----------------------------------------------------------------
            # LOAD FORECAST FILE
            # -----------------------------------------------------------------
            # -----------------------------------------------------------------
            # LOAD FORECAST FILE
            # -----------------------------------------------------------------

            forecast_df = pd.read_csv(forecast_path)

            print(f"Loaded rows: {len(forecast_df)}")

            # -----------------------------------------------------------------
            # HANDLE POSSIBLE MISSING / DIFFERENT COLUMN NAMES
            # -----------------------------------------------------------------

            # Strip spaces from column names first
            forecast_df.columns = [str(c).strip() for c in forecast_df.columns]

            print("Original columns:")
            print(forecast_df.columns.tolist())

            # -------------------------------------------------------------
            # HANDLE DATE COLUMN
            # -------------------------------------------------------------

            if "Date" not in forecast_df.columns:

                print("Date column header missing - renaming first column to 'Date'")

                cols = list(forecast_df.columns)
                cols[0] = "Date"

                forecast_df.columns = cols

            # -------------------------------------------------------------
            # HANDLE ENTRIES COLUMN
            # -------------------------------------------------------------

            entries_col = None

            possible_entries_cols = [
                "Entries",
                "entry_forecast"
            ]

            for col in possible_entries_cols:
                if col in forecast_df.columns:
                    entries_col = col
                    break

            if entries_col is None:
                print("No entries forecast column found")
                entries_accuracy_results[target_date_only][lead] = np.nan
                exits_accuracy_results[target_date_only][lead] = np.nan
                continue

            # -------------------------------------------------------------
            # HANDLE EXITS COLUMN
            # -------------------------------------------------------------

            exits_col = None

            possible_exits_cols = [
                "Exits",
                "exit_forecast"
            ]

            for col in possible_exits_cols:
                if col in forecast_df.columns:
                    exits_col = col
                    break

            if exits_col is None:
                print("No exits forecast column found")
                entries_accuracy_results[target_date_only][lead] = np.nan
                exits_accuracy_results[target_date_only][lead] = np.nan
                continue

            print(f"Using entries column: {entries_col}")
            print(f"Using exits column:   {exits_col}")

            # -----------------------------------------------------------------
            # CLEAN DATE
            # -----------------------------------------------------------------
            date_sample = str(forecast_df["Date"].dropna().iloc[0])

            print(f"Sample date value: {date_sample}")

            # Detect format automatically
            if "-" in date_sample:
                
                # Likely YYYY-MM-DD
                forecast_df["Date"] = pd.to_datetime(
                    forecast_df["Date"],
                    format="%Y-%m-%d",
                    errors="coerce"
                )

            else:
                
                # Likely DD/MM/YYYY
                forecast_df["Date"] = pd.to_datetime(
                    forecast_df["Date"],
                    format="%d/%m/%Y",
                    errors="coerce"
                )

            # Debugging
            if forecast_df["Date"].isna().all():
                print("All forecast dates failed to parse")
                entries_accuracy_results[target_date_only][lead] = np.nan
                exits_accuracy_results[target_date_only][lead] = np.nan
                continue
            # -----------------------------------------------------------------
            # FILTER TO TARGET DATE
            # -----------------------------------------------------------------

            target_forecast = forecast_df[
                forecast_df["Date"].dt.date == target_date_only
            ]

            if target_forecast.empty:
                print("No matching forecast rows for target date")
                entries_accuracy_results[target_date_only][lead] = np.nan
                exits_accuracy_results[target_date_only][lead] = np.nan
                continue

            # -----------------------------------------------------------------
            # DAILY FORECAST TOTALS
            # -----------------------------------------------------------------

            forecast_entries = pd.to_numeric(
                target_forecast[entries_col],
                errors="coerce"
            ).fillna(0).sum()

            forecast_exits = pd.to_numeric(
                target_forecast[exits_col],
                errors="coerce"
            ).fillna(0).sum()

            print(f"Forecast Entries: {forecast_entries:.2f}")
            print(f"Forecast Exits:   {forecast_exits:.2f}")

            print(f"Actual Entries:   {actual_entries}")
            print(f"Actual Exits:     {actual_exits}")

            # Store forecasts
            entries_forecast_results[target_date_only][lead] = forecast_entries
            exits_forecast_results[target_date_only][lead] = forecast_exits

            # Store actuals
            entries_actual_results[target_date_only][lead] = actual_entries
            exits_actual_results[target_date_only][lead] = actual_exits


            # -----------------------------------------------------------------
            # ACCURACY CALC
            # Formula:
            # (Actual - Forecast) / Actual
            # -----------------------------------------------------------------

            if actual_entries != 0:
                entries_accuracy = 1-abs((
                    (actual_entries - forecast_entries)
                    / actual_entries
                ))
            else:
                entries_accuracy = np.nan

            if actual_exits != 0:
                exits_accuracy = 1-abs((
                    (actual_exits - forecast_exits)
                    / actual_exits
                ))
            else:
                exits_accuracy = np.nan

            entries_accuracy_results[target_date_only][lead] = entries_accuracy
            exits_accuracy_results[target_date_only][lead] = exits_accuracy

            print(f"Entries Accuracy: {entries_accuracy:.4f}")
            print(f"Exits Accuracy:   {exits_accuracy:.4f}")

        except Exception as e:

            print(f"ERROR reading forecast file:")
            print(f"{forecast_filename}")
            print(str(e))

            entries_accuracy_results[target_date_only][lead] = np.nan
            exits_accuracy_results[target_date_only][lead] = np.nan

# -----------------------------------------------------------------------------
# FINAL DATAFRAMES
# -----------------------------------------------------------------------------

entries_accuracy_df = pd.DataFrame.from_dict(
    entries_accuracy_results,
    orient="index"
)

exits_accuracy_df = pd.DataFrame.from_dict(
    exits_accuracy_results,
    orient="index"
)

entries_accuracy_df.index.name = "Date"
exits_accuracy_df.index.name = "Date"

entries_accuracy_df = entries_accuracy_df.sort_index()
exits_accuracy_df = exits_accuracy_df.sort_index()

lead_name_map = {
    42: "6_weeks_out",
    35: "5_weeks_out",
    28: "4_weeks_out",
    21: "3_weeks_out",
    14: "2_weeks_out",
    7: "1_week_out"
}

# Rename columns
entries_accuracy_df.columns = [
    lead_name_map[c]
    for c in entries_accuracy_df.columns
]

exits_accuracy_df.columns = [
    lead_name_map[c]
    for c in exits_accuracy_df.columns
]

# -----------------------------------------------------------------------------
# OUTPUT
# -----------------------------------------------------------------------------

print("\n================ ENTRIES ACCURACY ================\n")
print(entries_accuracy_df)

print("\n================ EXITS ACCURACY ================\n")
print(exits_accuracy_df)

# -----------------------------------------------------------------------------
# FORECAST VALUE TABLES
# -----------------------------------------------------------------------------

entries_forecast_df = pd.DataFrame.from_dict(
    entries_forecast_results,
    orient="index"
)

exits_forecast_df = pd.DataFrame.from_dict(
    exits_forecast_results,
    orient="index"
)

entries_forecast_df.index.name = "Date"
exits_forecast_df.index.name = "Date"

entries_forecast_df = entries_forecast_df.sort_index()
exits_forecast_df = exits_forecast_df.sort_index()

entries_forecast_df.columns = [
    lead_name_map[c]
    for c in entries_forecast_df.columns
]

exits_forecast_df.columns = [
    lead_name_map[c]
    for c in exits_forecast_df.columns
]

# -----------------------------------------------------------------------------
# ACTUAL VALUE TABLES
# -----------------------------------------------------------------------------

entries_actual_df = pd.DataFrame.from_dict(
    entries_actual_results,
    orient="index"
)

exits_actual_df = pd.DataFrame.from_dict(
    exits_actual_results,
    orient="index"
)

entries_actual_df.index.name = "Date"
exits_actual_df.index.name = "Date"

entries_actual_df = entries_actual_df.sort_index()
exits_actual_df = exits_actual_df.sort_index()

entries_actual_df.columns = [
    lead_name_map[c] for c in entries_actual_df.columns
]

exits_actual_df.columns = [
    lead_name_map[c] for c in exits_actual_df.columns
]

# -----------------------------------------------------------------------------
# PRINT OUTPUTS
# -----------------------------------------------------------------------------

print("\n================ ENTRIES FORECASTS ================\n")
print(entries_forecast_df)

print("\n================ EXITS FORECASTS ================\n")
print(exits_forecast_df)

print("\n================ ENTRIES ACTUALS ================\n")
print(entries_actual_df)

print("\n================ EXITS ACTUALS ================\n")
print(exits_actual_df)

# -----------------------------------------------------------------------------
# MONTHLY ACCURACY SUMMARY
# -----------------------------------------------------------------------------

entries_monthly = entries_accuracy_df.copy()
entries_monthly.index = pd.to_datetime(entries_monthly.index)

entries_monthly["Month"] = (
    entries_monthly.index.to_period("M")
)

monthly_entries_accuracy = (
    entries_monthly
    .groupby("Month")
    .mean()
)

print("\n================ MONTHLY ENTRIES ACCURACY ================\n")
print((monthly_entries_accuracy * 100).round(1))


exits_monthly = exits_accuracy_df.copy()
exits_monthly.index = pd.to_datetime(exits_monthly.index)

exits_monthly["Month"] = (
    exits_monthly.index.to_period("M")
)

monthly_exits_accuracy = (
    exits_monthly
    .groupby("Month")
    .mean()
)

print("\n================ MONTHLY EXITS ACCURACY ================\n")
print((monthly_exits_accuracy * 100).round(1))

Analysis period:
Start: 2026-03-29
End:   2026-10-24


C:\Users\jamie_douglas\AppData\Local\Temp\ipykernel_6556\2363101416.py:58: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fastpark_actual_entries_df = pd.read_sql(sql, connect)
C:\Users\jamie_douglas\AppData\Local\Temp\ipykernel_6556\2363101416.py:70: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fastpark_actual_exits_df = pd.read_sql(sql, connect)



Actuals daily:
        Date  ActualEntries  ActualExits
0 2026-03-29            407          434
1 2026-03-30            418          449
2 2026-03-31            338          363
3 2026-04-01            530          334
4 2026-04-02            596          292

Checking forecast file:
Target date: 2026-03-29
Lead time: 42
Forecast file: 2026-02-15_daily_forecast.csv
Loaded rows: 58
Original columns:
['Unnamed: 0', 'booking_entries', 'booking_exits', 'entry_forecast', 'exit_forecast']
Date column header missing - renaming first column to 'Date'
Using entries column: entry_forecast
Using exits column:   exit_forecast
Sample date value: 2026-02-16
Forecast Entries: 493.00
Forecast Exits:   564.00
Actual Entries:   407
Actual Exits:     434
Entries Accuracy: 0.7887
Exits Accuracy:   0.7005

Checking forecast file:
Target date: 2026-03-29
Lead time: 35
Forecast file: 2026-02-22_daily_forecast.csv
Loaded rows: 58
Original columns:
['Unnamed: 0', 'booking_entries', 'booking_exits', 'entry_fo

save to excel

In [31]:
# -----------------------------------------------------------------------------
# SAVE TO EXCEL
# -----------------------------------------------------------------------------

output_folder = Path(
    rf"C:\Users\{user}\OneDrive - Edinburgh Airport Limited\Documents\Fast Park Report\Summer Wash-up 2026"
)

output_folder.mkdir(parents=True, exist_ok=True)

output_file = output_folder / (
    f"forecast_accuracy_{pd.Timestamp.today().strftime('%d%m%y')}.xlsx"
)

# -----------------------------------------------------------------------------
# WRITE TO EXCEL
# -----------------------------------------------------------------------------

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    sheet_name = "Forecast Accuracy"

    # -------------------------------------------------------------------------
    # ENTRIES SECTION
    # -------------------------------------------------------------------------

    current_row = 0

    entries_forecast_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=0
    )

    worksheet = writer.sheets[sheet_name]

    worksheet.cell(
        row=current_row + 1,
        column=1
    ).value = "Entries Forecast"

    current_row += len(entries_forecast_df) + 4

    entries_actual_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=0
    )

    worksheet.cell(
        row=current_row + 1,
        column=1
    ).value = "Entries Actuals"

    current_row += len(entries_actual_df) + 4

    entries_accuracy_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=0
    )

    worksheet.cell(
        row=current_row + 1,
        column=1
    ).value = "Entries Accuracy"

    # -------------------------------------------------------------------------
    # EXITS SECTION
    # -------------------------------------------------------------------------

    exits_start_col = len(entries_forecast_df.columns) + 4

    current_row = 0

    exits_forecast_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=exits_start_col
    )

    worksheet.cell(
        row=current_row + 1,
        column=exits_start_col + 1
    ).value = "Exits Forecast"

    current_row += len(exits_forecast_df) + 4

    exits_actual_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=exits_start_col
    )

    worksheet.cell(
        row=current_row + 1,
        column=exits_start_col + 1
    ).value = "Exits Actuals"

    current_row += len(exits_actual_df) + 4

    exits_accuracy_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=exits_start_col
    )

    worksheet.cell(
        row=current_row + 1,
        column=exits_start_col + 1
    ).value = "Exits Accuracy"

    # -------------------------------------------------------------------------
    # MONTHLY SUMMARY SHEETS
    # -------------------------------------------------------------------------

    monthly_entries_accuracy.to_excel(
        writer,
        sheet_name="Monthly Entries Accuracy"
    )

    monthly_exits_accuracy.to_excel(
        writer,
        sheet_name="Monthly Exits Accuracy"
    )

    monthly_entry_ws = writer.sheets["Monthly Entries Accuracy"]
    monthly_exit_ws = writer.sheets["Monthly Exits Accuracy"]

    # -------------------------------------------------------------------------
    # FORMATTING
    # -------------------------------------------------------------------------

    from openpyxl.styles import Font

    bold_font = Font(bold=True)

    # Main sheet titles
    for row in worksheet.iter_rows():
        for cell in row:
            if cell.value in [
                "Entries Forecast",
                "Entries Actuals",
                "Entries Accuracy",
                "Exits Forecast",
                "Exits Actuals",
                "Exits Accuracy"
            ]:
                cell.font = bold_font

    # Monthly sheet titles
    monthly_entry_ws["A1"].font = bold_font
    monthly_exit_ws["A1"].font = bold_font

    # Monthly sheets as %
    for ws in [monthly_entry_ws, monthly_exit_ws]:

        for row in ws.iter_rows(
            min_row=2,
            min_col=2
        ):
            for cell in row:

                if isinstance(cell.value, (int, float)):
                    cell.number_format = "0.0%"

    # Auto-width all sheets
    for ws in [worksheet, monthly_entry_ws, monthly_exit_ws]:

        for column_cells in ws.columns:

            length = 0
            column = column_cells[0].column_letter

            for cell in column_cells:

                try:
                    length = max(
                        length,
                        len(str(cell.value))
                    )
                except:
                    pass

            ws.column_dimensions[column].width = length + 3

print(f"\nExcel saved:")
print(output_file)


Excel saved:
C:\Users\jamie_douglas\OneDrive - Edinburgh Airport Limited\Documents\Fast Park Report\Summer Wash-up 2026\forecast_accuracy_041126.xlsx


## Forecast Accuracy Visual Story (Slide-Ready)
These visuals are structured to answer:
1. How accuracy evolves from 6 weeks out to 1 week out, month by month.
2. What factors are most associated with inaccuracy.
3. How June compares with September and October.

In [32]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -----------------------------------------------------------------------------
# VISUAL CONFIG
# -----------------------------------------------------------------------------

FONT_FAMILY = "Aptos, Segoe UI, Arial"
BASE_LAYOUT = dict(
    template="simple_white",
    font=dict(family=FONT_FAMILY, size=14, color="#1F2937"),
    title_font=dict(size=24, color="#0F172A"),
    margin=dict(l=60, r=30, t=90, b=60),
    plot_bgcolor="#FAFAF8",
    paper_bgcolor="#FAFAF8"
 )

LEAD_ORDER = [
    "6_weeks_out",
    "5_weeks_out",
    "4_weeks_out",
    "3_weeks_out",
    "2_weeks_out",
    "1_week_out"
]

LEAD_LABELS = {
    "6_weeks_out": "6W",
    "5_weeks_out": "5W",
    "4_weeks_out": "4W",
    "3_weeks_out": "3W",
    "2_weeks_out": "2W",
    "1_week_out": "1W"
}

MONTH_HIGHLIGHTS = ["2026-06", "2026-09", "2026-10"]
HIGHLIGHT_COLORS = {
    "2026-06": "#0B6E4F",
    "2026-09": "#D95F02",
    "2026-10": "#6A4C93"
}

def monthly_accuracy_long(df: pd.DataFrame, metric_name: str) -> pd.DataFrame:
    tmp = df.copy()
    if isinstance(tmp.index, pd.PeriodIndex):
        tmp.index = tmp.index.to_timestamp()
    else:
        tmp.index = pd.to_datetime(tmp.index)
    ordered_cols = [c for c in LEAD_ORDER if c in tmp.columns]
    tmp = tmp[ordered_cols]
    long_df = (
        tmp.reset_index()
        .rename(columns={"index": "Month"})
        .melt(id_vars="Month", var_name="Lead", value_name="Accuracy")
    )
    long_df["Month"] = pd.to_datetime(long_df["Month"]).dt.to_period("M").astype(str)
    long_df["LeadLabel"] = long_df["Lead"].map(LEAD_LABELS)
    long_df["AccuracyPct"] = long_df["Accuracy"] * 100
    long_df["Metric"] = metric_name
    return long_df.dropna(subset=["AccuracyPct"])

entries_monthly_long = monthly_accuracy_long(monthly_entries_accuracy, "Entries")
exits_monthly_long = monthly_accuracy_long(monthly_exits_accuracy, "Exits")
monthly_long = pd.concat([entries_monthly_long, exits_monthly_long], ignore_index=True)

available_months = sorted(monthly_long["Month"].unique().tolist())
available_highlights = [m for m in MONTH_HIGHLIGHTS if m in available_months]

print("Prepared monthly tables for visualisation.")
print(f"Months available: {available_months}")
print(f"Highlighted months used: {available_highlights}")

Prepared monthly tables for visualisation.
Months available: ['2026-03', '2026-04', '2026-05', '2026-06', '2026-07', '2026-08', '2026-09', '2026-10']
Highlighted months used: ['2026-06', '2026-09', '2026-10']


### 1) Accuracy Progression: 6 Weeks Out to 1 Week Out (Monthly Context)

In [33]:
plot_df = monthly_long.copy()

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Entries", "Exits"),
    shared_yaxes=True
)

for col_idx, metric in enumerate(["Entries", "Exits"], start=1):
    metric_df = plot_df[plot_df["Metric"] == metric]

    all_months = sorted(metric_df["Month"].unique().tolist())
    baseline_months = [m for m in all_months if m not in available_highlights]

    for month in baseline_months:
        part = metric_df[metric_df["Month"] == month]
        fig.add_trace(
            go.Scatter(
                x=part["LeadLabel"],
                y=part["AccuracyPct"],
                mode="lines+markers",
                line=dict(color="#CBD5E1", width=1.5),
                marker=dict(size=6, color="#CBD5E1"),
                name=f"{month} (context)",
                legendgroup="context",
                showlegend=(col_idx == 1 and month == baseline_months[0])
            ),
            row=1,
            col=col_idx
        )

    for month in available_highlights:
        part = metric_df[metric_df["Month"] == month]
        if part.empty:
            continue
        fig.add_trace(
            go.Scatter(
                x=part["LeadLabel"],
                y=part["AccuracyPct"],
                mode="lines+markers",
                line=dict(color=HIGHLIGHT_COLORS[month], width=4),
                marker=dict(size=10, color=HIGHLIGHT_COLORS[month]),
                name=month,
                legendgroup=month,
                showlegend=(col_idx == 1)
            ),
            row=1,
            col=col_idx
        )

fig.update_layout(
    **BASE_LAYOUT,
    title="Forecast Accuracy Progression by Lead Time (Monthly)",
    height=560,
    legend_title_text="Month",
    hovermode="x unified"
 )

fig.update_xaxes(
    categoryorder="array",
    categoryarray=[LEAD_LABELS[c] for c in LEAD_ORDER],
    title_text="Forecast Version (Weeks Before Arrival)"
 )
fig.update_yaxes(title_text="Accuracy (%)", range=[55, 100])

fig.show()

### 2) Monthly Heatmap: Where Accuracy Is Strong or Weak

In [34]:
def build_heatmap_matrix(monthly_df: pd.DataFrame) -> pd.DataFrame:
    tmp = monthly_df.copy()
    if isinstance(tmp.index, pd.PeriodIndex):
        tmp.index = tmp.index.astype(str)
    else:
        tmp.index = pd.to_datetime(tmp.index).to_period("M").astype(str)
    ordered_cols = [c for c in LEAD_ORDER if c in tmp.columns]
    tmp = (tmp[ordered_cols] * 100).rename(columns=LEAD_LABELS)
    return tmp

entries_heat = build_heatmap_matrix(monthly_entries_accuracy)
exits_heat = build_heatmap_matrix(monthly_exits_accuracy)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Entries Accuracy", "Exits Accuracy"),
    horizontal_spacing=0.12
)

fig.add_trace(
    go.Heatmap(
        z=entries_heat.values,
        x=entries_heat.columns,
        y=entries_heat.index,
        colorscale=[
            [0.0, "#B23A48"],
            [0.5, "#F4C95D"],
            [1.0, "#2A9D8F"]
        ],
        zmin=60,
        zmax=100,
        text=np.round(entries_heat.values, 1),
        texttemplate="%{text}",
        textfont=dict(size=11),
        colorbar=dict(title="Accuracy %")
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Heatmap(
        z=exits_heat.values,
        x=exits_heat.columns,
        y=exits_heat.index,
        colorscale=[
            [0.0, "#B23A48"],
            [0.5, "#F4C95D"],
            [1.0, "#2A9D8F"]
        ],
        zmin=60,
        zmax=100,
        text=np.round(exits_heat.values, 1),
        texttemplate="%{text}",
        textfont=dict(size=11),
        showscale=False
    ),
    row=1,
    col=2
)

fig.update_layout(
    **BASE_LAYOUT,
    title="Monthly Accuracy Heatmap by Forecast Version",
    height=560
)

fig.update_xaxes(title_text="Forecast Version")
fig.update_yaxes(title_text="Arrival Month")

fig.show()

### 3) June vs September vs October: Benchmark and Variability

In [35]:
focus_leads = ["6_weeks_out", "3_weeks_out", "1_week_out"]

focus_df = monthly_long[
    monthly_long["Month"].isin(available_highlights)
    & monthly_long["Lead"].isin(focus_leads)
]

summary = (
    focus_df.groupby(["Metric", "Month", "LeadLabel"], as_index=False)["AccuracyPct"]
    .mean()
    .rename(columns={"AccuracyPct": "MeanAccuracy"})
)

# Build daily long-format accuracy directly from accuracy tables
def build_daily_accuracy_long(accuracy_df: pd.DataFrame, metric_name: str) -> pd.DataFrame:
    tmp = accuracy_df.copy()
    tmp.index = pd.to_datetime(tmp.index)
    cols = [c for c in LEAD_ORDER if c in tmp.columns]
    out = (
        tmp[cols]
        .reset_index()
        .rename(columns={"index": "Date"})
        .melt(id_vars="Date", var_name="Lead", value_name="Accuracy")
    )
    out["Month"] = out["Date"].dt.to_period("M").astype(str)
    out["LeadLabel"] = out["Lead"].map(LEAD_LABELS)
    out["AccuracyPct"] = out["Accuracy"] * 100
    out["Metric"] = metric_name
    return out.dropna(subset=["AccuracyPct"])

entries_daily_accuracy = build_daily_accuracy_long(entries_accuracy_df, "Entries")
exits_daily_accuracy = build_daily_accuracy_long(exits_accuracy_df, "Exits")
daily_accuracy_long = pd.concat([entries_daily_accuracy, exits_daily_accuracy], ignore_index=True)

daily_focus = daily_accuracy_long[
    daily_accuracy_long["Month"].isin(available_highlights)
    & daily_accuracy_long["Lead"].isin(focus_leads)
]
variability = (
    daily_focus.groupby(["Metric", "Month", "LeadLabel"], as_index=False)["AccuracyPct"]
    .std()
    .rename(columns={"AccuracyPct": "StdDev"})
)

summary = summary.merge(variability, on=["Metric", "Month", "LeadLabel"], how="left")
summary["Label"] = summary["Month"] + " | " + summary["LeadLabel"]

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Entries", "Exits"),
    shared_yaxes=True
)

for i, metric in enumerate(["Entries", "Exits"], start=1):
    part = summary[summary["Metric"] == metric].copy()
    part = part.sort_values(["Month", "LeadLabel"])
    colors = [HIGHLIGHT_COLORS.get(m, "#64748B") for m in part["Month"]]

    fig.add_trace(
        go.Bar(
            x=part["Label"],
            y=part["MeanAccuracy"],
            marker_color=colors,
            error_y=dict(type="data", array=part["StdDev"].fillna(0), visible=True),
            text=part["MeanAccuracy"].round(1),
            textposition="outside",
            name=metric,
            showlegend=False
        ),
        row=1,
        col=i
    )

fig.update_layout(
    **BASE_LAYOUT,
    title="June Benchmark vs September/October Variability",
    height=560
 )

fig.update_yaxes(title_text="Mean Accuracy (%)", range=[55, 100])
fig.update_xaxes(title_text="Month | Forecast Version")

fig.show()

# Distribution view for variability context
fig_box = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Entries Distribution", "Exits Distribution"),
    shared_yaxes=True
)

for i, metric in enumerate(["Entries", "Exits"], start=1):
    metric_part = daily_focus[daily_focus["Metric"] == metric]
    for month in available_highlights:
        month_part = metric_part[metric_part["Month"] == month]
        if month_part.empty:
            continue
        fig_box.add_trace(
            go.Box(
                x=month_part["LeadLabel"],
                y=month_part["AccuracyPct"],
                name=month,
                marker_color=HIGHLIGHT_COLORS[month],
                boxmean=True,
                legendgroup=month,
                showlegend=(i == 1)
            ),
            row=1,
            col=i
        )

fig_box.update_layout(
    **BASE_LAYOUT,
    title="Accuracy Distribution: June vs September vs October",
    height=560,
    boxmode="group"
 )
fig_box.update_yaxes(title_text="Accuracy (%)", range=[40, 100])
fig_box.update_xaxes(title_text="Forecast Version")

fig_box.show()

## 4) Summer comparison (Apr-Oct): R1 vs R3 vs Actuals vs 6W/3W/1W forecasts

This section compares R1 and R3 separately (not combined) against Actuals and short-term forecast counts at both monthly and weekly levels for summer months (April to October).

In [36]:
# -----------------------------------------------------------------------------
# SUMMER COMPARISON DATA PREP (APR-OCT)
# R1 (own) vs R3 (own) vs Actuals vs 6W/3W/1W forecasts
# Includes nearest-run fallback when exact lead-day forecast file is missing
# -----------------------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np

washup_input_dir = Path(r"C:\Users\jamie_douglas\OneDrive - Edinburgh Airport Limited\Documents\GitHub\EDI_airport_analytics\inputs\FastPark S26 Wash-up")

r1_df = pd.read_csv(washup_input_dir / "R1_transaction_forecast.csv")
r3_df = pd.read_csv(washup_input_dir / "R3_transaction_forecast.csv")
entries_profile = pd.read_csv(washup_input_dir / "hourly_entries_by_dow.csv")
exits_profile = pd.read_csv(washup_input_dir / "hourly_exits_by_dow.csv")

SUMMER_MONTHS = [4, 5, 6, 7, 8, 9, 10]
LEAD_MAP = {42: "6w", 21: "3w", 7: "1w"}
MAX_FALLBACK_DAYS = 14


def normalise_transaction_forecast(df: pd.DataFrame, src_name: str) -> pd.DataFrame:
    df = df.copy()
    txn_col = "Transactions" if "Transactions" in df.columns else "Transcations"

    month_parsed = pd.to_datetime(df["Month"], format="%b-%y", errors="coerce")
    if month_parsed.isna().all():
        month_parsed = pd.to_datetime(df["Month"], format="%y-%b", errors="coerce")

    if month_parsed.isna().any():
        raise ValueError(f"Could not parse month values in {src_name} transaction file")

    return pd.DataFrame({
        "Month": month_parsed.dt.to_period("M"),
        "transactions": pd.to_numeric(df[txn_col], errors="coerce").fillna(0.0)
    })


def prep_profile(df: pd.DataFrame) -> pd.DataFrame:
    prof = df.copy()
    prof["Month"] = prof["Month"].astype(str).str.zfill(2)
    for h in [str(i) for i in range(24)]:
        prof[h] = pd.to_numeric(prof[h], errors="coerce").fillna(0.0)
    return prof


def allocate_product_to_daily(
    monthly_df: pd.DataFrame,
    product_label: str,
    entries_prof: pd.DataFrame,
    exits_prof: pd.DataFrame
) -> pd.DataFrame:
    hour_cols = [str(i) for i in range(24)]
    records = []

    for _, row in monthly_df.iterrows():
        month_period = row["Month"]
        month_total = float(row["transactions"])
        month_start = month_period.to_timestamp(how="start")
        month_end = month_period.to_timestamp(how="end")
        month_id = month_start.strftime("%m")

        for dt in pd.date_range(month_start, month_end, freq="D"):
            dow = dt.day_name()

            e_row = entries_prof[(entries_prof["Month"] == month_id) & (entries_prof["DayOfWeek"] == dow)]
            x_row = exits_prof[(exits_prof["Month"] == month_id) & (exits_prof["DayOfWeek"] == dow)]

            if e_row.empty or x_row.empty:
                continue

            e_weights = e_row.iloc[0][hour_cols].to_numpy(dtype=float)
            x_weights = x_row.iloc[0][hour_cols].to_numpy(dtype=float)

            for hour in range(24):
                records.append({
                    "Product": product_label,
                    "Month": month_period,
                    "Date": dt.normalize(),
                    "entry_weight": e_weights[hour],
                    "exit_weight": x_weights[hour],
                    "transactions": month_total
                })

    hourly = pd.DataFrame(records)
    if hourly.empty:
        raise ValueError(f"No hourly allocation rows generated for {product_label}")

    hourly["entry_weight_sum"] = hourly.groupby(["Product", "Month"], observed=False)["entry_weight"].transform("sum")
    hourly["exit_weight_sum"] = hourly.groupby(["Product", "Month"], observed=False)["exit_weight"].transform("sum")

    entry_col = f"{product_label.lower()}_entries"
    exit_col = f"{product_label.lower()}_exits"

    hourly[entry_col] = np.where(
        hourly["entry_weight_sum"] > 0,
        hourly["transactions"] * hourly["entry_weight"] / hourly["entry_weight_sum"],
        0.0
    )
    hourly[exit_col] = np.where(
        hourly["exit_weight_sum"] > 0,
        hourly["transactions"] * hourly["exit_weight"] / hourly["exit_weight_sum"],
        0.0
    )

    daily = (
        hourly
        .groupby(["Month", "Date"], observed=False)[[entry_col, exit_col]]
        .sum()
        .reset_index()
    )

    return daily


def get_forecast_file_candidates(run_date: pd.Timestamp) -> list[Path]:
    date_str = run_date.strftime("%Y-%m-%d")
    return [
        forecast_folder / f"{date_str}_daily_forecast.csv",
        forecast_folder / f"{date_str}_daily_forecast - Copy.csv"
    ]


forecast_cache: dict[pd.Timestamp, pd.DataFrame | None] = {}


def load_forecast_df(run_date: pd.Timestamp) -> pd.DataFrame | None:
    run_date = pd.Timestamp(run_date).normalize()
    if run_date in forecast_cache:
        return forecast_cache[run_date]

    for fp in get_forecast_file_candidates(run_date):
        if fp.exists():
            try:
                df = pd.read_csv(fp)
                df.columns = [str(c).strip() for c in df.columns]

                if "Date" not in df.columns:
                    cols = list(df.columns)
                    cols[0] = "Date"
                    df.columns = cols

                date_sample = str(df["Date"].dropna().iloc[0]) if not df["Date"].dropna().empty else ""
                if "-" in date_sample:
                    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d", errors="coerce")
                else:
                    df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y", errors="coerce")

                entries_col = "Entries" if "Entries" in df.columns else ("entry_forecast" if "entry_forecast" in df.columns else None)
                exits_col = "Exits" if "Exits" in df.columns else ("exit_forecast" if "exit_forecast" in df.columns else None)

                if entries_col is None or exits_col is None:
                    forecast_cache[run_date] = None
                    return None

                clean = df[["Date", entries_col, exits_col]].rename(columns={
                    entries_col: "EntriesForecast",
                    exits_col: "ExitsForecast"
                })
                forecast_cache[run_date] = clean
                return clean
            except Exception:
                forecast_cache[run_date] = None
                return None

    forecast_cache[run_date] = None
    return None


def fallback_offsets(max_days: int) -> list[int]:
    # Prefer exact match, then nearest by absolute distance.
    # Tie-break rule: prefer closer-to-arrival run first (+d), then farther-out run (-d).
    offsets = [0]
    for d in range(1, max_days + 1):
        offsets.extend([d, -d])
    return offsets


def get_forecast_for_target_with_fallback(target_date: pd.Timestamp, desired_lead_days: int) -> dict:
    target_date = pd.Timestamp(target_date).normalize()
    desired_run = target_date - pd.Timedelta(days=desired_lead_days)

    for off in fallback_offsets(MAX_FALLBACK_DAYS):
        candidate_run = desired_run + pd.Timedelta(days=off)
        fdf = load_forecast_df(candidate_run)
        if fdf is None:
            continue

        row = fdf[fdf["Date"].dt.normalize() == target_date]
        if row.empty:
            continue

        actual_lead = int((target_date - candidate_run).days)
        entries_val = pd.to_numeric(row["EntriesForecast"], errors="coerce").fillna(0).sum()
        exits_val = pd.to_numeric(row["ExitsForecast"], errors="coerce").fillna(0).sum()

        return {
            "entries": float(entries_val),
            "exits": float(exits_val),
            "run_date": candidate_run,
            "lead_used": actual_lead,
            "lead_offset": actual_lead - desired_lead_days
        }

    return {
        "entries": np.nan,
        "exits": np.nan,
        "run_date": pd.NaT,
        "lead_used": np.nan,
        "lead_offset": np.nan
    }


entries_profile = prep_profile(entries_profile)
exits_profile = prep_profile(exits_profile)

r1_monthly = normalise_transaction_forecast(r1_df, "R1")
r3_monthly = normalise_transaction_forecast(r3_df, "R3")

r1_daily = allocate_product_to_daily(r1_monthly, "R1", entries_profile, exits_profile)
r3_daily = allocate_product_to_daily(r3_monthly, "R3", entries_profile, exits_profile)

r_daily = (
    r1_daily
    .merge(r3_daily, on=["Month", "Date"], how="outer")
    .fillna(0.0)
)

r_daily["Date"] = pd.to_datetime(r_daily["Date"])
r_daily["MonthLabel"] = r_daily["Month"].astype(str)
r_daily["WeekStart"] = (r_daily["Date"] - pd.to_timedelta(r_daily["Date"].dt.weekday, unit="D")).dt.normalize()

target_dates = sorted(r_daily["Date"].dropna().unique().tolist())
forecast_rows = []

for dt in target_dates:
    dt_ts = pd.Timestamp(dt).normalize()
    row = {"Date": dt_ts}

    for desired_lead, short_name in LEAD_MAP.items():
        res = get_forecast_for_target_with_fallback(dt_ts, desired_lead)
        row[f"entries_{short_name}"] = res["entries"]
        row[f"exits_{short_name}"] = res["exits"]
        row[f"lead_used_{short_name}"] = res["lead_used"]
        row[f"lead_offset_{short_name}"] = res["lead_offset"]

    forecast_rows.append(row)

forecast_fallback_daily = pd.DataFrame(forecast_rows)

actual_daily_cmp = actuals_daily.copy()
actual_daily_cmp["Date"] = pd.to_datetime(actual_daily_cmp["Date"])

summer_daily = (
    r_daily
    .merge(forecast_fallback_daily, on="Date", how="left")
    .merge(actual_daily_cmp[["Date", "ActualEntries", "ActualExits"]], on="Date", how="left")
)

summer_daily = summer_daily[summer_daily["Date"].dt.month.isin(SUMMER_MONTHS)].copy()

monthly_summer_comparison = (
    summer_daily
    .groupby(["Month", "MonthLabel"], observed=False)[[
        "r1_entries", "r3_entries", "entries_6w", "entries_3w", "entries_1w", "ActualEntries",
        "r1_exits", "r3_exits", "exits_6w", "exits_3w", "exits_1w", "ActualExits"
    ]]
    .sum(min_count=1)
    .reset_index()
    .sort_values("Month")
)

weekly_summer_comparison = (
    summer_daily
    .groupby("WeekStart", observed=False)[[
        "r1_entries", "r3_entries", "entries_6w", "entries_3w", "entries_1w", "ActualEntries",
        "r1_exits", "r3_exits", "exits_6w", "exits_3w", "exits_1w", "ActualExits"
    ]]
    .sum(min_count=1)
    .reset_index()
    .sort_values("WeekStart")
)

cov = []
for short_name, label in [("6w", "6W"), ("3w", "3W"), ("1w", "1W")]:
    ser = summer_daily[f"entries_{short_name}"]
    used = summer_daily[f"lead_used_{short_name}"]
    offs = summer_daily[f"lead_offset_{short_name}"]
    cov.append({
        "Lead": label,
        "CoverageDays": int(ser.notna().sum()),
        "TotalDays": int(len(summer_daily)),
        "CoveragePct": round(100 * ser.notna().mean(), 1) if len(summer_daily) > 0 else np.nan,
        "AvgLeadUsed": round(float(pd.to_numeric(used, errors="coerce").mean()), 2) if used.notna().any() else np.nan,
        "AvgLeadOffset": round(float(pd.to_numeric(offs, errors="coerce").mean()), 2) if offs.notna().any() else np.nan
    })
coverage_diagnostics = pd.DataFrame(cov)

print("Summer months included:", SUMMER_MONTHS)
print("\nCoverage diagnostics after nearest-run fallback:")
print(coverage_diagnostics.to_string(index=False))
print("\nMonthly summer comparison:")
print(monthly_summer_comparison.round(1))
print("\nWeekly summer comparison (first 20 rows):")
print(weekly_summer_comparison.round(1).head(20))

Summer months included: [4, 5, 6, 7, 8, 9, 10]

Coverage diagnostics after nearest-run fallback:
Lead  CoverageDays  TotalDays  CoveragePct  AvgLeadUsed  AvgLeadOffset
  6W           214        214        100.0        41.94          -0.06
  3W           214        214        100.0        20.95          -0.05
  1W           214        214        100.0         6.95          -0.05

Monthly summer comparison:
     Month MonthLabel  r1_entries  r3_entries  entries_6w  entries_3w  \
0  2026-04    2026-04     15688.0     14586.0     17169.0     17765.0   
1  2026-05    2026-05     15014.7     16434.0     18961.0     17856.0   
2  2026-06    2026-06     16678.9     16103.0     17354.0     17980.0   
3  2026-07    2026-07     16114.9     14994.0     18199.0     14967.0   
4  2026-08    2026-08     17422.4     16315.0     14330.0     14064.0   
5  2026-09    2026-09     18865.5     20211.0     16046.0     19081.0   
6  2026-10    2026-10     19069.1     20008.0     16610.0     20685.0   

   ent

In [37]:
# -----------------------------------------------------------------------------
# SUMMER COMPARISON VISUALS (MONTHLY + WEEKLY)
# -----------------------------------------------------------------------------

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---- Monthly ----
fig_m = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Monthly summer comparison - Entries",
        "Monthly summer comparison - Exits"
    ),
    horizontal_spacing=0.12
)

# Actuals as bars
fig_m.add_trace(
    go.Bar(
        x=monthly_summer_comparison["MonthLabel"],
        y=monthly_summer_comparison["ActualEntries"],
        name="Actuals",
        marker_color="#9CA3AF",
        opacity=0.45
    ),
    row=1,
    col=1
)

fig_m.add_trace(
    go.Bar(
        x=monthly_summer_comparison["MonthLabel"],
        y=monthly_summer_comparison["ActualExits"],
        name="Actuals",
        marker_color="#9CA3AF",
        opacity=0.45,
        showlegend=False
    ),
    row=1,
    col=2
)

monthly_series_entries = [
    ("r1_entries", "R1", "#8E44AD"),
    ("r3_entries", "R3", "#E67E22"),
    ("entries_6w", "Forecast 6W", "#1F77B4"),
    ("entries_3w", "Forecast 3W", "#2CA02C"),
    ("entries_1w", "Forecast 1W", "#D62728")
]

for col, name, color in monthly_series_entries:
    fig_m.add_trace(
        go.Scatter(
            x=monthly_summer_comparison["MonthLabel"],
            y=monthly_summer_comparison[col],
            mode="lines+markers",
            name=name,
            line=dict(width=2.5, color=color),
            marker=dict(size=7)
        ),
        row=1,
        col=1
    )

monthly_series_exits = [
    ("r1_exits", "R1", "#8E44AD"),
    ("r3_exits", "R3", "#E67E22"),
    ("exits_6w", "Forecast 6W", "#1F77B4"),
    ("exits_3w", "Forecast 3W", "#2CA02C"),
    ("exits_1w", "Forecast 1W", "#D62728")
]

for col, name, color in monthly_series_exits:
    fig_m.add_trace(
        go.Scatter(
            x=monthly_summer_comparison["MonthLabel"],
            y=monthly_summer_comparison[col],
            mode="lines+markers",
            name=name,
            line=dict(width=2.5, color=color),
            marker=dict(size=7),
            showlegend=False
        ),
        row=1,
        col=2
    )

fig_m.update_layout(
    **BASE_LAYOUT,
    title="Summer Monthly (Apr-Oct): R1 vs R3 vs Actuals vs 6W/3W/1W Forecasts",
    height=560,
    width=1360,
    barmode="overlay",
    legend=dict(orientation="h", y=1.1, x=0)
)
fig_m.update_yaxes(title_text="Count", row=1, col=1)
fig_m.update_yaxes(title_text="Count", row=1, col=2)
fig_m.update_xaxes(title_text="Month", row=1, col=1)
fig_m.update_xaxes(title_text="Month", row=1, col=2)

fig_m.show()

# ---- Weekly ----
fig_w = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Weekly summer comparison - Entries",
        "Weekly summer comparison - Exits"
    ),
    horizontal_spacing=0.12
)

# Actuals as bars
fig_w.add_trace(
    go.Bar(
        x=weekly_summer_comparison["WeekStart"],
        y=weekly_summer_comparison["ActualEntries"],
        name="Actuals",
        marker_color="#9CA3AF",
        opacity=0.4
    ),
    row=1,
    col=1
)

fig_w.add_trace(
    go.Bar(
        x=weekly_summer_comparison["WeekStart"],
        y=weekly_summer_comparison["ActualExits"],
        name="Actuals",
        marker_color="#9CA3AF",
        opacity=0.4,
        showlegend=False
    ),
    row=1,
    col=2
)

weekly_series_entries = [
    ("r1_entries", "R1", "#8E44AD"),
    ("r3_entries", "R3", "#E67E22"),
    ("entries_6w", "Forecast 6W", "#1F77B4"),
    ("entries_3w", "Forecast 3W", "#2CA02C"),
    ("entries_1w", "Forecast 1W", "#D62728")
]

for col, name, color in weekly_series_entries:
    fig_w.add_trace(
        go.Scatter(
            x=weekly_summer_comparison["WeekStart"],
            y=weekly_summer_comparison[col],
            mode="lines+markers",
            name=name,
            line=dict(width=2, color=color),
            marker=dict(size=6)
        ),
        row=1,
        col=1
    )

weekly_series_exits = [
    ("r1_exits", "R1", "#8E44AD"),
    ("r3_exits", "R3", "#E67E22"),
    ("exits_6w", "Forecast 6W", "#1F77B4"),
    ("exits_3w", "Forecast 3W", "#2CA02C"),
    ("exits_1w", "Forecast 1W", "#D62728")
]

for col, name, color in weekly_series_exits:
    fig_w.add_trace(
        go.Scatter(
            x=weekly_summer_comparison["WeekStart"],
            y=weekly_summer_comparison[col],
            mode="lines+markers",
            name=name,
            line=dict(width=2, color=color),
            marker=dict(size=6),
            showlegend=False
        ),
        row=1,
        col=2
    )

fig_w.update_layout(
    **BASE_LAYOUT,
    title="Summer Weekly (Apr-Oct): R1 vs R3 vs Actuals vs 6W/3W/1W Forecasts",
    height=560,
    width=1360,
    barmode="overlay",
    legend=dict(orientation="h", y=1.1, x=0)
)
fig_w.update_yaxes(title_text="Count", row=1, col=1)
fig_w.update_yaxes(title_text="Count", row=1, col=2)
fig_w.update_xaxes(title_text="Week commencing", row=1, col=1)
fig_w.update_xaxes(title_text="Week commencing", row=1, col=2)

fig_w.show()

In [38]:
# Quick diagnostic: June bias check and metric confirmation (fallback-adjusted)
june_period = pd.Period("2026-06", freq="M")

print("Accuracy metric formula used in core section:")
print("accuracy = 1 - abs((actual - forecast) / actual)")

print("\nSummer comparison section uses fallback-adjusted lead forecasts from forecast_fallback_daily.")

if "monthly_summer_comparison" in globals() and not monthly_summer_comparison.empty:
    june = monthly_summer_comparison[monthly_summer_comparison["Month"] == june_period]
    if not june.empty:
        cols = [
            "MonthLabel",
            "ActualEntries", "entries_6w", "entries_3w", "entries_1w", "r1_entries", "r3_entries",
            "ActualExits", "exits_6w", "exits_3w", "exits_1w", "r1_exits", "r3_exits"
        ]
        print("\nJune monthly comparison row (fallback-adjusted):")
        print(june[cols].round(1).to_string(index=False))

        for lead_col, actual_col, label in [
            ("entries_6w", "ActualEntries", "Entries 6W"),
            ("entries_3w", "ActualEntries", "Entries 3W"),
            ("entries_1w", "ActualEntries", "Entries 1W"),
            ("exits_6w", "ActualExits", "Exits 6W"),
            ("exits_3w", "ActualExits", "Exits 3W"),
            ("exits_1w", "ActualExits", "Exits 1W")
        ]:
            actual = float(june.iloc[0][actual_col])
            forecast = float(june.iloc[0][lead_col])
            delta = forecast - actual
            pct = (delta / actual) * 100 if actual != 0 else np.nan
            direction = "under" if delta < 0 else "over"
            print(f"{label}: forecast-actual={delta:.1f} ({pct:.1f}%), {direction}-forecast")
    else:
        print("No June row found in monthly_summer_comparison.")
else:
    print("monthly_summer_comparison is not available. Run the summer prep cell first.")

if "monthly_entries_accuracy" in globals() and "monthly_exits_accuracy" in globals():
    if june_period in monthly_entries_accuracy.index and june_period in monthly_exits_accuracy.index:
        print("\nJune accuracy % from core accuracy tables:")
        print((monthly_entries_accuracy.loc[june_period] * 100).round(1).to_string())
        print((monthly_exits_accuracy.loc[june_period] * 100).round(1).to_string())

Accuracy metric formula used in core section:
accuracy = 1 - abs((actual - forecast) / actual)

Summer comparison section uses fallback-adjusted lead forecasts from forecast_fallback_daily.

June monthly comparison row (fallback-adjusted):
MonthLabel  ActualEntries  entries_6w  entries_3w  entries_1w  r1_entries  r3_entries  ActualExits  exits_6w  exits_3w  exits_1w  r1_exits  r3_exits
   2026-06        16019.0     17354.0     17980.0     16904.0     16678.9     16103.0      15685.0   16630.0   16992.0   15401.0   16678.9   16103.0
Entries 6W: forecast-actual=1335.0 (8.3%), over-forecast
Entries 3W: forecast-actual=1961.0 (12.2%), over-forecast
Entries 1W: forecast-actual=885.0 (5.5%), over-forecast
Exits 6W: forecast-actual=945.0 (6.0%), over-forecast
Exits 3W: forecast-actual=1307.0 (8.3%), over-forecast
Exits 1W: forecast-actual=-284.0 (-1.8%), under-forecast

June accuracy % from core accuracy tables:
6_weeks_out    82.7
5_weeks_out    80.4
4_weeks_out    80.2
3_weeks_out    78.9
2

In [39]:
# Coverage diagnostic for June forecast-day completeness (fallback-adjusted)
june_start = pd.Timestamp("2026-06-01")
june_end = pd.Timestamp("2026-06-30")

if "summer_daily" in globals() and not summer_daily.empty:
    june = summer_daily[(summer_daily["Date"] >= june_start) & (summer_daily["Date"] <= june_end)].copy()

    print("June fallback-adjusted coverage (non-null days out of 30):")
    for short_name in ["6w", "3w", "1w"]:
        e_col = f"entries_{short_name}"
        x_col = f"exits_{short_name}"
        lead_col = f"lead_used_{short_name}"
        off_col = f"lead_offset_{short_name}"

        e_days = int(june[e_col].notna().sum()) if e_col in june.columns else 0
        x_days = int(june[x_col].notna().sum()) if x_col in june.columns else 0

        mean_lead = float(pd.to_numeric(june[lead_col], errors="coerce").mean()) if lead_col in june.columns else np.nan
        mean_off = float(pd.to_numeric(june[off_col], errors="coerce").mean()) if off_col in june.columns else np.nan

        print(f"{short_name}: entries={e_days}/30, exits={x_days}/30, avg lead used={mean_lead:.2f}, avg offset={mean_off:.2f}")
else:
    print("summer_daily not available. Run the summer prep cell first.")

June fallback-adjusted coverage (non-null days out of 30):
6w: entries=30/30, exits=30/30, avg lead used=42.00, avg offset=0.00
3w: entries=30/30, exits=30/30, avg lead used=20.90, avg offset=-0.10
1w: entries=30/30, exits=30/30, avg lead used=6.80, avg offset=-0.20


In [40]:
# June 1-week-out diagnostics: fallback-adjusted coverage and missing dates
import pandas as pd

june_start = pd.Timestamp("2026-06-01")
june_end = pd.Timestamp("2026-06-30")
all_june_dates = pd.date_range(june_start, june_end, freq="D")

if "summer_daily" in globals() and not summer_daily.empty:
    june = summer_daily[(summer_daily["Date"] >= june_start) & (summer_daily["Date"] <= june_end)].copy()

    entries_1w_series = june.set_index("Date")["entries_1w"] if "entries_1w" in june.columns else pd.Series(dtype=float)
    exits_1w_series = june.set_index("Date")["exits_1w"] if "exits_1w" in june.columns else pd.Series(dtype=float)

    missing_entries_dates = [d.strftime("%Y-%m-%d") for d in all_june_dates if d not in entries_1w_series.dropna().index]
    missing_exits_dates = [d.strftime("%Y-%m-%d") for d in all_june_dates if d not in exits_1w_series.dropna().index]

    entries_1w_sum = float(entries_1w_series.dropna().sum()) if len(entries_1w_series) else 0.0
    exits_1w_sum = float(exits_1w_series.dropna().sum()) if len(exits_1w_series) else 0.0
    entries_1w_days = int(entries_1w_series.notna().sum()) if len(entries_1w_series) else 0
    exits_1w_days = int(exits_1w_series.notna().sum()) if len(exits_1w_series) else 0

    entries_1w_mean = entries_1w_sum / entries_1w_days if entries_1w_days > 0 else float("nan")
    exits_1w_mean = exits_1w_sum / exits_1w_days if exits_1w_days > 0 else float("nan")

    print("June 1W totals currently used in monthly comparison (fallback-adjusted):")
    print(f"Entries 1W sum: {entries_1w_sum:.1f} across {entries_1w_days} days")
    print(f"Exits   1W sum: {exits_1w_sum:.1f} across {exits_1w_days} days")

    print("\nMissing June dates for 1W entries:")
    print(missing_entries_dates)
    print("\nMissing June dates for 1W exits:")
    print(missing_exits_dates)

    if "lead_used_1w" in june.columns:
        print("\nLead-day details for 1W fallback in June:")
        print(june[["Date", "lead_used_1w", "lead_offset_1w"]].to_string(index=False))

    print("\nIf scaled to a 30-day month (for context only):")
    print(f"Entries 1W scaled: {(entries_1w_mean * 30):.1f}")
    print(f"Exits   1W scaled: {(exits_1w_mean * 30):.1f}")
else:
    print("summer_daily not available. Run the summer prep cell first.")

June 1W totals currently used in monthly comparison (fallback-adjusted):
Entries 1W sum: 16904.0 across 30 days
Exits   1W sum: 15401.0 across 30 days

Missing June dates for 1W entries:
[]

Missing June dates for 1W exits:
[]

Lead-day details for 1W fallback in June:
      Date  lead_used_1w  lead_offset_1w
2026-06-01           7.0             0.0
2026-06-02           7.0             0.0
2026-06-03           7.0             0.0
2026-06-04           7.0             0.0
2026-06-05           7.0             0.0
2026-06-06           7.0             0.0
2026-06-07           7.0             0.0
2026-06-08           7.0             0.0
2026-06-09           7.0             0.0
2026-06-10           8.0             1.0
2026-06-11           9.0             2.0
2026-06-12           4.0            -3.0
2026-06-13           5.0            -2.0
2026-06-14           6.0            -1.0
2026-06-15           7.0             0.0
2026-06-16           7.0             0.0
2026-06-17           8.0         

## 5) Executive Visual Pack: Why Accuracy Moves, and How to Improve It
These charts are purpose-built for slides and directly answer:
1. How forecast accuracy evolves from initial forecast to 1 week out across the year (monthly + trend).
2. Which factors drive inaccuracy (especially pricing where available).
3. Why June performed strongly vs September and October, and the opportunity gap versus R1.

In [43]:
# -----------------------------------------------------------------------------
# DAILY + WEEKLY DUAL-AXIS VISUALS (SUMMER 2026 ONLY)
# Left axis = signed error (Forecast - Actual), Right axis = Accuracy %
# -----------------------------------------------------------------------------

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

SUMMER_START = pd.Timestamp("2026-03-29")
SUMMER_END = pd.Timestamp("2026-10-24")
TARGET_ACCURACY = 90.0

STYLE = dict(
    template="simple_white",
    font=dict(family="Aptos Display, Aptos, Calibri", size=14, color="#0E2433"),
    title_font=dict(size=26, color="#0B132B"),
    paper_bgcolor="#F8FBFD",
    plot_bgcolor="#FFFFFF",
    margin=dict(l=70, r=60, t=95, b=70)
)

def _daily_metric_df(acc_df, forecast_df, actual_df, lead_col, metric_name):
    acc = acc_df.copy()
    acc.index = pd.to_datetime(acc.index)

    fc = forecast_df.copy()
    fc.index = pd.to_datetime(fc.index)

    ac = actual_df.copy()
    ac.index = pd.to_datetime(ac.index)

    out = pd.DataFrame(index=acc.index)
    out["AccuracyPct"] = acc[lead_col] * 100
    out["Forecast"] = fc[lead_col]
    out["Actual"] = ac[lead_col]
    out["SignedError"] = out["Forecast"] - out["Actual"]
    out["Date"] = out.index
    out = out[(out["Date"] >= SUMMER_START) & (out["Date"] <= SUMMER_END)].copy()
    out["Metric"] = metric_name
    return out.reset_index(drop=True)

def _to_weekly(df):
    tmp = df.copy()
    tmp["WeekStart"] = (tmp["Date"] - pd.to_timedelta(tmp["Date"].dt.weekday, unit="D")).dt.normalize()
    weekly = (
        tmp.groupby("WeekStart", as_index=False)[["Forecast", "Actual"]]
        .sum(min_count=1)
        .sort_values("WeekStart")
    )
    weekly["SignedError"] = weekly["Forecast"] - weekly["Actual"]
    weekly["AccuracyPct"] = np.where(
        weekly["Actual"] != 0,
        (1 - np.abs((weekly["Actual"] - weekly["Forecast"]) / weekly["Actual"])) * 100,
        np.nan
    )
    weekly["Date"] = weekly["WeekStart"]
    return weekly

def _aggregate(df, grain):
    if grain == "weekly":
        return _to_weekly(df)
    out = df[["Date", "Forecast", "Actual", "SignedError", "AccuracyPct"]].copy()
    out = out.sort_values("Date")
    return out

def plot_dual_axis_split(lead_col, lead_title, grain="daily"):
    ent_daily = _daily_metric_df(entries_accuracy_df, entries_forecast_df, entries_actual_df, lead_col, "Entries")
    exi_daily = _daily_metric_df(exits_accuracy_df, exits_forecast_df, exits_actual_df, lead_col, "Exits")

    ent = _aggregate(ent_daily, grain)
    exi = _aggregate(exi_daily, grain)

    x_label = "Week commencing" if grain == "weekly" else "Arrival date"
    title_grain = "Weekly" if grain == "weekly" else "Daily"
    left_axis_label = f"{title_grain} signed error (Forecast - Actual)"
    right_axis_label = f"{title_grain} accuracy (%)"

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=(f"Entries ({lead_title})", f"Exits ({lead_title})"),
        specs=[[{"secondary_y": True}, {"secondary_y": True}]],
        shared_yaxes=False
    )

    for i, (df, err_color, acc_color, name) in enumerate([
        (ent, "#2563EB", "#0F766E", "Entries"),
        (exi, "#DC2626", "#C2410C", "Exits")
    ], start=1):
        fig.add_trace(
            go.Bar(
                x=df["Date"],
                y=df["SignedError"],
                name=f"{name} {grain} error (Forecast-Actual)",
                marker_color=err_color,
                opacity=0.55,
                showlegend=(i == 1)
            ),
            row=1, col=i, secondary_y=False
        )

        fig.add_trace(
            go.Scatter(
                x=df["Date"],
                y=df["AccuracyPct"],
                mode="lines+markers",
                name=f"{name} {grain} accuracy %",
                line=dict(width=3, color=acc_color),
                marker=dict(size=7),
                showlegend=(i == 1)
            ),
            row=1, col=i, secondary_y=True
        )

        fig.add_trace(
            go.Scatter(
                x=df["Date"],
                y=np.full(len(df), TARGET_ACCURACY),
                mode="lines",
                name="Target accuracy (90%)",
                line=dict(width=2, color="#166534", dash="dash"),
                showlegend=(i == 1)
            ),
            row=1, col=i, secondary_y=True
        )

        fig.add_hline(
            y=0,
            line_width=1.2,
            line_color="#475569",
            row=1, col=i, secondary_y=False
        )

    fig.update_layout(
        **STYLE,
        title=f"Summer 2026 {title_grain} | {lead_title}: Signed Error vs Accuracy",
        height=600,
        barmode="relative",
        legend=dict(orientation="h", y=1.1, x=0)
    )

    fig.update_yaxes(title_text=left_axis_label, row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text=left_axis_label, row=1, col=2, secondary_y=False)
    fig.update_yaxes(title_text=right_axis_label, range=[40, 100], row=1, col=1, secondary_y=True)
    fig.update_yaxes(title_text=right_axis_label, range=[40, 100], row=1, col=2, secondary_y=True)
    fig.update_xaxes(title_text=x_label, row=1, col=1)
    fig.update_xaxes(title_text=x_label, row=1, col=2)

    fig.show()

def plot_dual_axis_total(lead_col, lead_title, grain="daily"):
    ent_daily = _daily_metric_df(entries_accuracy_df, entries_forecast_df, entries_actual_df, lead_col, "Entries")
    exi_daily = _daily_metric_df(exits_accuracy_df, exits_forecast_df, exits_actual_df, lead_col, "Exits")

    combined_daily = pd.DataFrame({
        "Date": ent_daily["Date"].values,
        "Forecast": ent_daily["Forecast"].values + exi_daily["Forecast"].values,
        "Actual": ent_daily["Actual"].values + exi_daily["Actual"].values
    })
    combined_daily["SignedError"] = combined_daily["Forecast"] - combined_daily["Actual"]
    combined_daily["AccuracyPct"] = np.where(
        combined_daily["Actual"] != 0,
        (1 - np.abs((combined_daily["Actual"] - combined_daily["Forecast"]) / combined_daily["Actual"])) * 100,
        np.nan
    )

    total_df = _aggregate(combined_daily, grain)

    x_label = "Week commencing" if grain == "weekly" else "Arrival date"
    title_grain = "Weekly" if grain == "weekly" else "Daily"
    left_axis_label = f"{title_grain} signed error (Forecast - Actual)"
    right_axis_label = f"{title_grain} accuracy (%)"

    fig = make_subplots(rows=1, cols=1, specs=[[{"secondary_y": True}]])

    fig.add_trace(
        go.Bar(
            x=total_df["Date"],
            y=total_df["SignedError"],
            name=f"Total {grain} error (Forecast-Actual)",
            marker_color="#4F46E5",
            opacity=0.6
        ),
        row=1, col=1, secondary_y=False
    )

    fig.add_trace(
        go.Scatter(
            x=total_df["Date"],
            y=total_df["AccuracyPct"],
            mode="lines+markers",
            name=f"Total {grain} accuracy %",
            line=dict(width=3, color="#0F766E"),
            marker=dict(size=8)
        ),
        row=1, col=1, secondary_y=True
    )

    fig.add_trace(
        go.Scatter(
            x=total_df["Date"],
            y=np.full(len(total_df), TARGET_ACCURACY),
            mode="lines",
            name="Target accuracy (90%)",
            line=dict(width=2, color="#166534", dash="dash")
        ),
        row=1, col=1, secondary_y=True
    )

    fig.add_hline(y=0, line_width=1.2, line_color="#475569", row=1, col=1, secondary_y=False)

    fig.update_layout(
        **STYLE,
        title=f"Summer 2026 {title_grain} | Total (Entries + Exits) {lead_title}: Signed Error vs Accuracy",
        height=560,
        barmode="relative",
        legend=dict(orientation="h", y=1.12, x=0)
    )
    fig.update_yaxes(title_text=left_axis_label, row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text=right_axis_label, range=[40, 100], row=1, col=1, secondary_y=True)
    fig.update_xaxes(title_text=x_label, row=1, col=1)

    fig.show()

In [42]:
# -----------------------------------------------------------------------------
# DAILY PACK
# -----------------------------------------------------------------------------

plot_dual_axis_split("3_weeks_out", "3-Week-Out", grain="daily")
plot_dual_axis_split("1_week_out", "1-Week-Out", grain="daily")
plot_dual_axis_total("3_weeks_out", "3-Week-Out", grain="daily")
plot_dual_axis_total("1_week_out", "1-Week-Out", grain="daily")

# -----------------------------------------------------------------------------
# WEEKLY PACK
# -----------------------------------------------------------------------------

plot_dual_axis_split("3_weeks_out", "3-Week-Out", grain="weekly")
plot_dual_axis_split("1_week_out", "1-Week-Out", grain="weekly")
plot_dual_axis_total("3_weeks_out", "3-Week-Out", grain="weekly")
plot_dual_axis_total("1_week_out", "1-Week-Out", grain="weekly")

## 6) Pricing Correlation Analysis (Tables First)
This section adds analysis-first pricing tables before visuals.

Scope:
1. Average price/day by car park for bookings entering on each date.
2. Average price/day by car park for bookings exiting on each date.
3. Entry/Exit date vs Created date price/day matrices by car park.
4. FastPark price premium vs other car parks compared against booking-curve shift, accuracy, and signed forecast error.

In [44]:
# -----------------------------------------------------------------------------
# PRICING BASE TABLES: ENTRY-DATE / EXIT-DATE PRICE-DAY + PRICE MATRICES
# -----------------------------------------------------------------------------

import pandas as pd
import numpy as np
import pyodbc

SUMMER_START = pd.Timestamp("2026-03-29")
SUMMER_END = pd.Timestamp("2026-10-24")
ASSETS = ["FastPark", "Long Stay", "Mid Stay", "Multi", "Plane Parking", "Terminal"]
COMPETITORS = [a for a in ASSETS if a != "FastPark"]

def _safe_read_sql(sql_text: str):
    global connect
    try:
        return pd.read_sql(sql_text, connect)
    except Exception:
        connect = pyodbc.connect('DSN=' + dsn + ';')
        return pd.read_sql(sql_text, connect)

asset_list_sql = ", ".join([f"'{x}'" for x in ASSETS])
sql_bookings = f"""
SELECT bookingId, createdAt, entryDate, exitDate, status, assetName, bookingTotal, productQuantity
FROM AirportX.v_Bookings
WHERE assetName IN ({asset_list_sql})
  AND entryDate >= '{SUMMER_START.strftime('%Y-%m-%d')}'
  AND entryDate < '{(SUMMER_END + pd.Timedelta(days=1)).strftime('%Y-%m-%d')}'
  AND status = 'B'
"""
bookings_px = _safe_read_sql(sql_bookings)

bookings_px["createdAt"] = pd.to_datetime(bookings_px["createdAt"], errors="coerce")
bookings_px["entryDate"] = pd.to_datetime(bookings_px["entryDate"], errors="coerce")
bookings_px["exitDate"] = pd.to_datetime(bookings_px["exitDate"], errors="coerce")

bookings_px["created_date"] = bookings_px["createdAt"].dt.normalize()
bookings_px["entry_date"] = bookings_px["entryDate"].dt.normalize()
bookings_px["exit_date"] = bookings_px["exitDate"].dt.normalize()

bookings_px["duration_days"] = (bookings_px["exitDate"] - bookings_px["entryDate"]).dt.total_seconds() / 86400
bookings_px["duration_days"] = bookings_px["duration_days"].clip(lower=1)
bookings_px["lead_days"] = (bookings_px["entry_date"] - bookings_px["created_date"]).dt.days
bookings_px["price_per_day"] = bookings_px["bookingTotal"] / bookings_px["duration_days"].replace(0, np.nan)

bookings_px = bookings_px[bookings_px["lead_days"].between(0, 120, inclusive="both")].copy()

# 1) Entry-day avg/median price per day by car park
entry_daily_price_long = (
    bookings_px.groupby(["entry_date", "assetName"], as_index=False)
    .agg(
        avg_price_per_day=("price_per_day", "mean"),
        median_price_per_day=("price_per_day", "median"),
        bookings=("bookingId", "nunique"),
        median_lead_days=("lead_days", "median"),
        median_duration_days=("duration_days", "median")
    )
)
entry_daily_price_wide = (
    entry_daily_price_long.pivot(index="entry_date", columns="assetName", values="median_price_per_day")
    .sort_index()
)

# 2) Exit-day avg/median price per day by car park
exit_daily_price_long = (
    bookings_px.groupby(["exit_date", "assetName"], as_index=False)
    .agg(
        avg_price_per_day=("price_per_day", "mean"),
        median_price_per_day=("price_per_day", "median"),
        bookings=("bookingId", "nunique"),
        median_lead_days=("lead_days", "median"),
        median_duration_days=("duration_days", "median")
    )
)
exit_daily_price_wide = (
    exit_daily_price_long.pivot(index="exit_date", columns="assetName", values="median_price_per_day")
    .sort_index()
)

# 3) Entry/Exit date vs created-date matrices (all assets stored in dicts)
entry_price_matrices = {}
exit_price_matrices = {}
for asset in ASSETS:
    a = bookings_px[bookings_px["assetName"] == asset].copy()
    entry_price_matrices[asset] = (
        a.groupby(["entry_date", "created_date"], as_index=False)["price_per_day"]
        .median()
        .pivot(index="entry_date", columns="created_date", values="price_per_day")
        .sort_index()
        .sort_index(axis=1)
    )
    exit_price_matrices[asset] = (
        a.groupby(["exit_date", "created_date"], as_index=False)["price_per_day"]
        .median()
        .pivot(index="exit_date", columns="created_date", values="price_per_day")
        .sort_index()
        .sort_index(axis=1)
    )

print("Loaded bookings for pricing analysis:", bookings_px.shape)
print("\nEntry-date daily median price/day table preview:")
print(entry_daily_price_wide.head())
print("\nExit-date daily median price/day table preview:")
print(exit_daily_price_wide.head())
print("\nExample matrix shapes (FastPark):")
print("Entry matrix:", entry_price_matrices["FastPark"].shape)
print("Exit matrix:", exit_price_matrices["FastPark"].shape)

C:\Users\jamie_douglas\AppData\Local\Temp\ipykernel_6556\1505820114.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_text, connect)
C:\Users\jamie_douglas\AppData\Local\Temp\ipykernel_6556\1505820114.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_text, connect)


Loaded bookings for pricing analysis: (364661, 14)

Entry-date daily median price/day table preview:
assetName    FastPark  Long Stay   Mid Stay      Multi  Plane Parking  \
entry_date                                                              
2026-03-29  11.077594  10.223832  12.239759  20.195073       5.918400   
2026-03-30  10.920072  10.318218  13.550380  23.054234       7.116610   
2026-03-31  10.836498  11.423810  13.455701  25.754931       7.007592   
2026-04-01  10.920000   9.844923  13.526805  24.634615       6.649888   
2026-04-02  11.400352   9.752569  12.969730  18.421776       7.882336   

assetName    Terminal  
entry_date             
2026-03-29  14.619511  
2026-03-30  17.139286  
2026-03-31  18.144340  
2026-04-01  17.381802  
2026-04-02  14.753964  

Exit-date daily median price/day table preview:
assetName    FastPark  Long Stay   Mid Stay      Multi  Plane Parking  \
exit_date                                                               
2026-03-29   9.990000  1

In [49]:
# -----------------------------------------------------------------------------
# FASTPARK PREMIUM VS INDIVIDUAL COMPETITORS + BOOKING CURVE / ACCURACY
# -----------------------------------------------------------------------------

# FastPark daily entry context
entry_fp = entry_daily_price_long[entry_daily_price_long["assetName"] == "FastPark"].copy()
entry_fp = entry_fp.rename(columns={
    "entry_date": "Date",
    "median_price_per_day": "fp_median_ppd",
    "avg_price_per_day": "fp_avg_ppd",
    "median_lead_days": "fp_median_lead_days",
    "median_duration_days": "fp_median_duration_days",
    "bookings": "fp_bookings"
})

# Competitor daily median price/day by asset (individual comparison base)
entry_comp_by_asset = (
    entry_daily_price_long[entry_daily_price_long["assetName"].isin(COMPETITORS)]
    [["entry_date", "assetName", "median_price_per_day"]]
    .pivot(index="entry_date", columns="assetName", values="median_price_per_day")
    .reset_index()
    .rename(columns={"entry_date": "Date"})
)

entry_signal = entry_fp.merge(entry_comp_by_asset, on="Date", how="left")

# FastPark premium/discount vs each competitor individually
for comp in COMPETITORS:
    col = f"premium_vs_{comp.lower().replace(' ', '_')}_pct"
    entry_signal[col] = (entry_signal["fp_median_ppd"] / entry_signal[comp] - 1) * 100

# Optional grouped comparison columns requested if lumping is needed
entry_signal["comp_terminal_multi_median_ppd"] = entry_signal[["Terminal", "Multi"]].median(axis=1)
entry_signal["comp_other3_median_ppd"] = entry_signal[["Long Stay", "Mid Stay", "Plane Parking"]].median(axis=1)
entry_signal["premium_vs_terminal_multi_pct"] = (entry_signal["fp_median_ppd"] / entry_signal["comp_terminal_multi_median_ppd"] - 1) * 100
entry_signal["premium_vs_other3_pct"] = (entry_signal["fp_median_ppd"] / entry_signal["comp_other3_median_ppd"] - 1) * 100

# Lead-time baseline by duration bucket (booking-curve shift metric)
tmp = bookings_px[bookings_px["assetName"] == "FastPark"].copy()
tmp["duration_bucket"] = pd.cut(
    tmp["duration_days"],
    bins=[0, 1, 3, 7, 10, 14, 21, np.inf],
    labels=["0-1", "2-3", "4-7", "8-10", "11-14", "15-21", "22+"],
    include_lowest=True
)
baseline_lead_by_duration = (
    tmp.groupby("duration_bucket", observed=False)["lead_days"]
    .median()
    .astype(float)
    .to_dict()
)
tmp["duration_bucket_key"] = tmp["duration_bucket"].astype(str)
tmp["duration_bucket_baseline"] = tmp["duration_bucket_key"].map(baseline_lead_by_duration)
tmp["lead_delta_vs_duration_baseline"] = tmp["lead_days"] - tmp["duration_bucket_baseline"]

entry_curve_shift = (
    tmp.groupby("entry_date", as_index=False)
    .agg(
        lead_delta_vs_duration_baseline=("lead_delta_vs_duration_baseline", "median"),
        fp_median_lead_days_raw=("lead_days", "median")
    )
    .rename(columns={"entry_date": "Date"})
)

entry_signal = entry_signal.merge(entry_curve_shift, on="Date", how="left")

# Accuracy and signed error series for 3W and 1W on target date
acc_tbl = pd.DataFrame({"Date": pd.to_datetime(entries_accuracy_df.index)})
acc_tbl = acc_tbl.sort_values("Date")

ea = entries_actual_df.copy(); ea.index = pd.to_datetime(ea.index)
xa = exits_actual_df.copy(); xa.index = pd.to_datetime(xa.index)
ef = entries_forecast_df.copy(); ef.index = pd.to_datetime(ef.index)
xf = exits_forecast_df.copy(); xf.index = pd.to_datetime(xf.index)

for d, c in {21: "3_weeks_out", 7: "1_week_out"}.items():
    actual_total = ea[c] + xa[c]
    forecast_total = ef[c] + xf[c]
    signed_err = forecast_total - actual_total
    total_acc = np.where(actual_total != 0, (1 - np.abs((actual_total - forecast_total) / actual_total)) * 100, np.nan)

    acc_tbl[f"total_acc_{d}d"] = acc_tbl["Date"].map(pd.Series(total_acc, index=actual_total.index))
    acc_tbl[f"signed_error_{d}d"] = acc_tbl["Date"].map(pd.Series(signed_err, index=actual_total.index))

entry_signal = entry_signal.merge(acc_tbl, on="Date", how="left")
entry_signal = entry_signal.sort_values("Date")

# Exit-date context with individual competitor columns
exit_fp = exit_daily_price_long[exit_daily_price_long["assetName"] == "FastPark"].copy()
exit_fp = exit_fp.rename(columns={
    "exit_date": "Date",
    "median_price_per_day": "fp_median_ppd",
    "avg_price_per_day": "fp_avg_ppd",
    "median_lead_days": "fp_median_lead_days",
    "median_duration_days": "fp_median_duration_days",
    "bookings": "fp_bookings"
})

exit_comp_by_asset = (
    exit_daily_price_long[exit_daily_price_long["assetName"].isin(COMPETITORS)]
    [["exit_date", "assetName", "median_price_per_day"]]
    .pivot(index="exit_date", columns="assetName", values="median_price_per_day")
    .reset_index()
    .rename(columns={"exit_date": "Date"})
)

exit_signal = exit_fp.merge(exit_comp_by_asset, on="Date", how="left")
for comp in COMPETITORS:
    col = f"premium_vs_{comp.lower().replace(' ', '_')}_pct"
    exit_signal[col] = (exit_signal["fp_median_ppd"] / exit_signal[comp] - 1) * 100

# Correlation summaries for individual competitor premiums
premium_cols = [f"premium_vs_{c.lower().replace(' ', '_')}_pct" for c in COMPETITORS]
entry_corr_cols = premium_cols + [
    "premium_vs_terminal_multi_pct",
    "premium_vs_other3_pct",
    "lead_delta_vs_duration_baseline",
    "fp_median_lead_days",
    "total_acc_21d",
    "total_acc_7d",
    "signed_error_21d",
    "signed_error_7d"
]
entry_corr = entry_signal[entry_corr_cols].corr(numeric_only=True)

print("Entry-signal table shape:", entry_signal.shape)
print("Exit-signal table shape:", exit_signal.shape)
print("\nEntry signal preview (individual competitor premiums):")
preview_cols = [
    "Date", "fp_median_ppd", "Long Stay", "Mid Stay", "Multi", "Plane Parking", "Terminal",
    "premium_vs_long_stay_pct", "premium_vs_mid_stay_pct", "premium_vs_multi_pct",
    "premium_vs_plane_parking_pct", "premium_vs_terminal_pct",
    "total_acc_21d", "total_acc_7d", "signed_error_21d", "signed_error_7d"
]
print(entry_signal[preview_cols].head(12).to_string(index=False))

print("\nCorrelation matrix (entry context, individual competitor premiums):")
print(entry_corr.round(3).to_string())

Entry-signal table shape: (210, 27)
Exit-signal table shape: (239, 17)

Entry signal preview (individual competitor premiums):
      Date  fp_median_ppd  Long Stay  Mid Stay     Multi  Plane Parking  Terminal  premium_vs_long_stay_pct  premium_vs_mid_stay_pct  premium_vs_multi_pct  premium_vs_plane_parking_pct  premium_vs_terminal_pct  total_acc_21d  total_acc_7d  signed_error_21d  signed_error_7d
2026-03-29      11.077594  10.223832 12.239759 20.195073       5.918400 14.619511                  8.350703                -9.494999            -45.147046                     87.172113               -24.227324      58.501784     89.060642             349.0             92.0
2026-03-30      10.920072  10.318218 13.550380 23.054234       7.116610 17.139286                  5.832926               -19.411326            -52.633119                     53.444847               -36.286309      65.167243     88.927336             302.0             96.0
2026-03-31      10.836498  11.423810 13.455701 25.7

In [50]:
# -----------------------------------------------------------------------------
# SEGMENTED TABLES: PER-COMPETITOR PREMIUM BINS + OVER/UNDER-FORECAST RATES
# -----------------------------------------------------------------------------

entry_segment = entry_signal.copy()
entry_segment = entry_segment[(entry_segment["Date"] >= SUMMER_START) & (entry_segment["Date"] <= SUMMER_END)].copy()

premium_cols = [f"premium_vs_{c.lower().replace(' ', '_')}_pct" for c in COMPETITORS]
summary_frames = []

for pcol in premium_cols:
    comp_name = pcol.replace("premium_vs_", "").replace("_pct", "").replace("_", " ").title()
    seg = entry_segment.copy()

    seg["premium_bin"] = pd.cut(
        seg[pcol],
        bins=[-np.inf, -10, -2, 2, 10, np.inf],
        labels=["<= -10%", "-10% to -2%", "-2% to +2%", "+2% to +10%", "> +10%"],
        include_lowest=True
    )

    seg["over_forecast_21d"] = seg["signed_error_21d"] > 0
    seg["under_forecast_21d"] = seg["signed_error_21d"] < 0
    seg["over_forecast_7d"] = seg["signed_error_7d"] > 0
    seg["under_forecast_7d"] = seg["signed_error_7d"] < 0

    out = (
        seg.groupby("premium_bin", observed=False)
        .agg(
            days=("Date", "count"),
            avg_premium_pct=(pcol, "mean"),
            median_premium_pct=(pcol, "median"),
            avg_acc_21d=("total_acc_21d", "mean"),
            avg_acc_7d=("total_acc_7d", "mean"),
            median_signed_error_21d=("signed_error_21d", "median"),
            median_signed_error_7d=("signed_error_7d", "median"),
            over_rate_21d=("over_forecast_21d", "mean"),
            under_rate_21d=("under_forecast_21d", "mean"),
            over_rate_7d=("over_forecast_7d", "mean"),
            under_rate_7d=("under_forecast_7d", "mean")
        )
        .reset_index()
    )

    for c in ["over_rate_21d", "under_rate_21d", "over_rate_7d", "under_rate_7d"]:
        out[c] = out[c] * 100

    out.insert(0, "competitor", comp_name)
    summary_frames.append(out)

premium_bin_summary_by_comp = pd.concat(summary_frames, ignore_index=True)

print("Premium-bin summary by competitor:")
print(premium_bin_summary_by_comp.round(2).to_string(index=False))

# Day-level comparison preview with individual competitor premium columns
entry_day_compare = entry_segment[[
    "Date", "fp_median_ppd", "Long Stay", "Mid Stay", "Multi", "Plane Parking", "Terminal",
    "premium_vs_long_stay_pct", "premium_vs_mid_stay_pct", "premium_vs_multi_pct",
    "premium_vs_plane_parking_pct", "premium_vs_terminal_pct",
    "total_acc_21d", "total_acc_7d", "signed_error_21d", "signed_error_7d"
]].sort_values("Date")

print("\nDay-level comparison preview (per competitor):")
print(entry_day_compare.head(15).to_string(index=False))

Premium-bin summary by competitor:
   competitor premium_bin  days  avg_premium_pct  median_premium_pct  avg_acc_21d  avg_acc_7d  median_signed_error_21d  median_signed_error_7d  over_rate_21d  under_rate_21d  over_rate_7d  under_rate_7d
    Long Stay     <= -10%    51           -17.15              -16.16        78.90       92.61                     93.0                    47.0          52.94           31.37         66.67          17.65
    Long Stay -10% to -2%    66            -6.15               -6.13        88.11       92.66                     43.5                    34.5          48.48           33.33         50.00          25.76
    Long Stay  -2% to +2%    32            -0.27               -0.62        84.76       92.31                     46.5                    20.0          46.88           34.38         56.25          40.62
    Long Stay +2% to +10%    36             5.67                5.00        83.88       92.05                    136.5                     3.0          6

In [52]:
# -----------------------------------------------------------------------------
# GRAPH: FASTPARK PREMIUM VS FORECAST ACCURACY (7D) BY INDIVIDUAL COMPETITOR
# -----------------------------------------------------------------------------

import plotly.graph_objects as go
from plotly.subplots import make_subplots

plot_parts = []
for comp in COMPETITORS:
    pcol = f"premium_vs_{comp.lower().replace(' ', '_')}_pct"
    part = entry_signal[["Date", pcol, "total_acc_7d", "total_acc_21d", "signed_error_7d", "signed_error_21d"]].copy()
    part = part.rename(columns={pcol: "premium_pct"})
    part["competitor"] = comp
    part = part.dropna(subset=["premium_pct", "total_acc_7d"])
    plot_parts.append(part)

premium_accuracy_long = pd.concat(plot_parts, ignore_index=True)

comp_stats = (
    premium_accuracy_long.groupby("competitor", as_index=False)
    .agg(
        corr_with_acc_7d=("premium_pct", lambda s: s.corr(premium_accuracy_long.loc[s.index, "total_acc_7d"])),
        obs=("premium_pct", "count")
    )
)

stat_map = {
    r["competitor"]: (r["corr_with_acc_7d"], int(r["obs"]))
    for _, r in comp_stats.iterrows()
}

titles = []
for comp in COMPETITORS:
    corr, obs = stat_map.get(comp, (np.nan, 0))
    if pd.notna(corr):
        titles.append(f"{comp}<br>r={corr:.2f}, n={obs}")
    else:
        titles.append(f"{comp}<br>r=NA, n={obs}")

fig_comp = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=titles,
    horizontal_spacing=0.08,
    vertical_spacing=0.16,
    shared_yaxes=False,
)

palette = {
    "Long Stay": "#1f77b4",
    "Mid Stay": "#2ca02c",
    "Multi": "#ff7f0e",
    "Plane Parking": "#d62728",
    "Terminal": "#9467bd",
}

for i, comp in enumerate(COMPETITORS):
    r = i // 3 + 1
    c = i % 3 + 1
    part = premium_accuracy_long[premium_accuracy_long["competitor"] == comp].copy()

    fig_comp.add_trace(
        go.Scatter(
            x=part["premium_pct"],
            y=part["total_acc_7d"],
            mode="markers",
            marker=dict(size=7, color=palette.get(comp, "#444"), opacity=0.65),
            name=comp,
            showlegend=False,
            hovertemplate=(
                "Date=%{customdata[0]|%Y-%m-%d}<br>"
                "Premium=%{x:.2f}%<br>"
                "Accuracy 7d=%{y:.1f}%<br>"
                "Accuracy 21d=%{customdata[1]:.1f}%<br>"
                "Signed error 7d=%{customdata[2]:.0f}<br>"
                "Signed error 21d=%{customdata[3]:.0f}<extra></extra>"
            ),
            customdata=part[["Date", "total_acc_21d", "signed_error_7d", "signed_error_21d"]].to_numpy(),
        ),
        row=r,
        col=c,
    )

    # Add a simple linear fit per competitor for slide readability
    if len(part) >= 5 and part["premium_pct"].nunique() > 1:
        x = part["premium_pct"].to_numpy()
        y = part["total_acc_7d"].to_numpy()
        m, b = np.polyfit(x, y, 1)
        xx = np.linspace(np.nanmin(x), np.nanmax(x), 100)
        yy = m * xx + b
        fig_comp.add_trace(
            go.Scatter(
                x=xx,
                y=yy,
                mode="lines",
                line=dict(color=palette.get(comp, "#444"), width=2),
                showlegend=False,
                hoverinfo="skip",
            ),
            row=r,
            col=c,
        )

    fig_comp.add_hline(y=90, line_width=1, line_dash="dash", line_color="#111", row=r, col=c)

fig_comp.update_xaxes(title_text="FastPark premium vs competitor (%)")
fig_comp.update_yaxes(title_text="Forecast accuracy (7d, %)")
fig_comp.update_layout(
    title="FastPark Price Premium vs Forecast Accuracy (1 Week Out) by Competitor",
    template="plotly_white",
    height=820,
    width=1250,
    margin=dict(l=50, r=30, t=80, b=60),
)

fig_comp.show()

In [55]:
# -----------------------------------------------------------------------------
# ADDED SET: 3W + 1W, ACCURACY % + FORECAST - ACTUALS (PER COMPETITOR)
# -----------------------------------------------------------------------------

import plotly.graph_objects as go
from plotly.subplots import make_subplots

palette = {
    "Long Stay": "#1f77b4",
    "Mid Stay": "#2ca02c",
    "Multi": "#ff7f0e",
    "Plane Parking": "#d62728",
    "Terminal": "#9467bd",
}

def plot_premium_vs_metric(metric_col, metric_label, title_text, ref_line=None):
    plot_parts = []
    for comp in COMPETITORS:
        pcol = f"premium_vs_{comp.lower().replace(' ', '_')}_pct"
        cols = [
            "Date", pcol, metric_col,
            "total_acc_7d", "total_acc_21d", "signed_error_7d", "signed_error_21d"
        ]
        cols = list(dict.fromkeys(cols))
        part = entry_signal[cols].copy()
        part = part.rename(columns={pcol: "premium_pct"})
        part["competitor"] = comp
        part = part.dropna(subset=["premium_pct", metric_col])
        plot_parts.append(part)

    d = pd.concat(plot_parts, ignore_index=True)

    stats_rows = []
    for comp in COMPETITORS:
        g = d[d["competitor"] == comp]
        stats_rows.append({
            "competitor": comp,
            "corr": g["premium_pct"].corr(g[metric_col]),
            "obs": int(g["premium_pct"].notna().sum()),
        })
    comp_stats = pd.DataFrame(stats_rows)

    stat_map = {
        r["competitor"]: (r["corr"], int(r["obs"]))
        for _, r in comp_stats.iterrows()
    }

    titles = []
    for comp in COMPETITORS:
        corr, obs = stat_map.get(comp, (np.nan, 0))
        if pd.notna(corr):
            titles.append(f"{comp}<br>r={corr:.2f}, n={obs}")
        else:
            titles.append(f"{comp}<br>r=NA, n={obs}")

    fig = make_subplots(
        rows=2,
        cols=3,
        subplot_titles=titles,
        horizontal_spacing=0.08,
        vertical_spacing=0.16,
        shared_yaxes=False,
    )

    for i, comp in enumerate(COMPETITORS):
        r = i // 3 + 1
        c = i % 3 + 1
        part = d[d["competitor"] == comp].copy()

        fig.add_trace(
            go.Scatter(
                x=part["premium_pct"],
                y=part[metric_col],
                mode="markers",
                marker=dict(size=7, color=palette.get(comp, "#444"), opacity=0.65),
                name=comp,
                showlegend=False,
                hovertemplate=(
                    "Date=%{customdata[0]|%Y-%m-%d}<br>"
                    "Premium=%{x:.2f}%<br>"
                    f"{metric_label}=%{{y:.2f}}<br>"
                    "Acc 7d=%{customdata[1]:.1f}%<br>"
                    "Acc 21d=%{customdata[2]:.1f}%<br>"
                    "Forecast - actuals 7d=%{customdata[3]:.0f}<br>"
                    "Forecast - actuals 21d=%{customdata[4]:.0f}<extra></extra>"
                ),
                customdata=part[["Date", "total_acc_7d", "total_acc_21d", "signed_error_7d", "signed_error_21d"]].to_numpy(),
            ),
            row=r,
            col=c,
        )

        if len(part) >= 5 and part["premium_pct"].nunique() > 1:
            x = part["premium_pct"].to_numpy()
            y = part[metric_col].to_numpy()
            m, b = np.polyfit(x, y, 1)
            xx = np.linspace(np.nanmin(x), np.nanmax(x), 100)
            yy = m * xx + b
            fig.add_trace(
                go.Scatter(
                    x=xx,
                    y=yy,
                    mode="lines",
                    line=dict(color=palette.get(comp, "#444"), width=2),
                    showlegend=False,
                    hoverinfo="skip",
                ),
                row=r,
                col=c,
            )

        if ref_line is not None:
            fig.add_hline(y=ref_line, line_width=1, line_dash="dash", line_color="#111", row=r, col=c)

    fig.update_xaxes(title_text="FastPark premium vs competitor (%)")
    fig.update_yaxes(title_text=metric_label)
    fig.update_layout(
        title=title_text,
        template="plotly_white",
        height=820,
        width=1250,
        margin=dict(l=50, r=30, t=80, b=60),
    )

    fig.show()

# 1) 3 weeks out: Forecast accuracy (%)
plot_premium_vs_metric(
    metric_col="total_acc_21d",
    metric_label="Forecast accuracy (3W, %)",
    title_text="FastPark Premium vs Forecast Accuracy (3 Weeks Out) by Competitor",
    ref_line=90,
 )

# 2) 1 week out: Forecast accuracy (%)
plot_premium_vs_metric(
    metric_col="total_acc_7d",
    metric_label="Forecast accuracy (1W, %)",
    title_text="FastPark Premium vs Forecast Accuracy (1 Week Out) by Competitor",
    ref_line=90,
 )

# 3) 3 weeks out: Forecast - actuals
plot_premium_vs_metric(
    metric_col="signed_error_21d",
    metric_label="Forecast - actuals (3W)",
    title_text="FastPark Premium vs Forecast - Actuals (3 Weeks Out) by Competitor",
    ref_line=0,
 )

# 4) 1 week out: Forecast - actuals
plot_premium_vs_metric(
    metric_col="signed_error_7d",
    metric_label="Forecast - actuals (1W)",
    title_text="FastPark Premium vs Forecast - Actuals (1 Week Out) by Competitor",
    ref_line=0,
 )

## 7) Re-show Original Daily/Weekly Accuracy Views + Pricing Event Diagnosis

> Re-run the original daily/weekly views (unchanged), then identify dates where accuracy drops align with significant FastPark relative price drops vs specific competitors.

In [56]:
# Re-show original views (unchanged) after pricing graphs
plot_dual_axis_split("3_weeks_out", "3-Week-Out", grain="daily")
plot_dual_axis_split("1_week_out", "1-Week-Out", grain="daily")
plot_dual_axis_total("3_weeks_out", "3-Week-Out", grain="daily")
plot_dual_axis_total("1_week_out", "1-Week-Out", grain="daily")

plot_dual_axis_split("3_weeks_out", "3-Week-Out", grain="weekly")
plot_dual_axis_split("1_week_out", "1-Week-Out", grain="weekly")
plot_dual_axis_total("3_weeks_out", "3-Week-Out", grain="weekly")
plot_dual_axis_total("1_week_out", "1-Week-Out", grain="weekly")

In [57]:
# -----------------------------------------------------------------------------
# EVENT DIAGNOSIS: ACCURACY DROPS VS COMPETITOR-SPECIFIC PREMIUM DROPS
# -----------------------------------------------------------------------------

# Build summer base
diag = entry_signal.copy()
diag = diag[(diag["Date"] >= SUMMER_START) & (diag["Date"] <= SUMMER_END)].copy()
diag = diag.sort_values("Date").reset_index(drop=True)

# Accuracy-drop definitions (target breach OR sudden decline vs trailing average)
diag["acc7d_roll7"] = diag["total_acc_7d"].rolling(7, min_periods=3).mean()
diag["acc21d_roll7"] = diag["total_acc_21d"].rolling(7, min_periods=3).mean()
diag["acc7d_drop_vs_roll"] = diag["total_acc_7d"] - diag["acc7d_roll7"]
diag["acc21d_drop_vs_roll"] = diag["total_acc_21d"] - diag["acc21d_roll7"]

diag["acc_drop_7d"] = (diag["total_acc_7d"] < 90) | (diag["acc7d_drop_vs_roll"] <= -5)
diag["acc_drop_21d"] = (diag["total_acc_21d"] < 90) | (diag["acc21d_drop_vs_roll"] <= -5)

# Competitor-specific premium drop event definitions
premium_cols = [f"premium_vs_{c.lower().replace(' ', '_')}_pct" for c in COMPETITORS]
comp_meta = []

for comp, pcol in zip(COMPETITORS, premium_cols):
    dcol = f"delta_{pcol}"
    diag[dcol] = diag[pcol].diff()

    # Threshold = more negative of fixed -7pp and data-driven mean-1.5*std
    mu = diag[dcol].mean(skipna=True)
    sd = diag[dcol].std(skipna=True)
    dyn_th = mu - 1.5 * sd if pd.notna(sd) else -7
    drop_th = min(-7, dyn_th)

    flag0 = f"drop_evt_{comp.lower().replace(' ', '_')}_lag0"
    diag[flag0] = diag[dcol] <= drop_th

    # Lags: did a significant premium drop happen in the prior 1/2/3 days?
    for lag in [1, 2, 3]:
        diag[f"drop_evt_{comp.lower().replace(' ', '_')}_lag{lag}"] = diag[flag0].shift(lag).fillna(False)

    diag[f"drop_evt_{comp.lower().replace(' ', '_')}_any_lag1_3"] = (
        diag[f"drop_evt_{comp.lower().replace(' ', '_')}_lag1"]
        | diag[f"drop_evt_{comp.lower().replace(' ', '_')}_lag2"]
        | diag[f"drop_evt_{comp.lower().replace(' ', '_')}_lag3"]
    )

    comp_meta.append({
        "competitor": comp,
        "premium_col": pcol,
        "delta_col": dcol,
        "drop_threshold": float(drop_th),
        "same_day_flag": flag0,
        "lag_any_flag": f"drop_evt_{comp.lower().replace(' ', '_')}_any_lag1_3",
    })

comp_thresholds = pd.DataFrame(comp_meta)

# Slide-ready date table: focus on dates with any 7d or 21d drop
drop_dates = diag[diag["acc_drop_7d"] | diag["acc_drop_21d"]].copy()

rows = []
for _, r in drop_dates.iterrows():
    for meta in comp_meta:
        comp = meta["competitor"]
        pcol = meta["premium_col"]
        dcol = meta["delta_col"]
        same_day = bool(r[meta["same_day_flag"]])
        lag_any = bool(r[meta["lag_any_flag"]])

        # Find nearest lag match (0..3) for narrative clarity
        nearest_lag = np.nan
        for lag in [0, 1, 2, 3]:
            flg_col = f"drop_evt_{comp.lower().replace(' ', '_')}_lag{lag}"
            if bool(r.get(flg_col, False)):
                nearest_lag = lag
                break

        rows.append({
            "Date": r["Date"],
            "competitor": comp,
            "accuracy_7d_pct": r["total_acc_7d"],
            "accuracy_21d_pct": r["total_acc_21d"],
            "forecast_minus_actuals_7d": r["signed_error_7d"],
            "forecast_minus_actuals_21d": r["signed_error_21d"],
            "premium_pct": r[pcol],
            "premium_delta_pp": r[dcol],
            "premium_drop_same_day": same_day,
            "premium_drop_prior_1_3_days": lag_any,
            "nearest_drop_lag_days": nearest_lag,
            "acc_drop_7d": bool(r["acc_drop_7d"]),
            "acc_drop_21d": bool(r["acc_drop_21d"]),
        })

drop_event_detail = pd.DataFrame(rows).sort_values(["Date", "competitor"])

# Hit-rate / lift summary for each competitor and horizon
lift_rows = []
for meta in comp_meta:
    comp = meta["competitor"]
    same_evt = diag[meta["same_day_flag"]].fillna(False)
    lag_evt = diag[meta["lag_any_flag"]].fillna(False)

    for horizon_label, drop_col in [("7d", "acc_drop_7d"), ("21d", "acc_drop_21d")]:
        y = diag[drop_col].fillna(False)

        for evt_label, evt in [("same_day", same_evt), ("prior_1_3_days", lag_evt)]:
            p_drop_given_evt = y[evt].mean() if evt.sum() > 0 else np.nan
            p_drop_given_noevt = y[~evt].mean() if (~evt).sum() > 0 else np.nan
            lift = (p_drop_given_evt / p_drop_given_noevt) if pd.notna(p_drop_given_evt) and pd.notna(p_drop_given_noevt) and p_drop_given_noevt > 0 else np.nan

            lift_rows.append({
                "competitor": comp,
                "horizon": horizon_label,
                "event_window": evt_label,
                "event_days": int(evt.sum()),
                "p_acc_drop_given_event": p_drop_given_evt,
                "p_acc_drop_given_no_event": p_drop_given_noevt,
                "lift_ratio": lift,
            })

event_lift_summary = pd.DataFrame(lift_rows).sort_values(["horizon", "event_window", "lift_ratio"], ascending=[True, True, False])

print("Competitor premium-drop thresholds (pp change in premium vs prior day):")
print(comp_thresholds.round(3).to_string(index=False))

print("\nDrop-date diagnostic table (first 40 rows):")
print(drop_event_detail.head(40).to_string(index=False))

print("\nEvent lift summary (higher lift => stronger association with accuracy drop):")
print(event_lift_summary.round(3).to_string(index=False))

Competitor premium-drop thresholds (pp change in premium vs prior day):
   competitor                  premium_col                          delta_col  drop_threshold               same_day_flag                      lag_any_flag
    Long Stay     premium_vs_long_stay_pct     delta_premium_vs_long_stay_pct         -16.480     drop_evt_long_stay_lag0     drop_evt_long_stay_any_lag1_3
     Mid Stay      premium_vs_mid_stay_pct      delta_premium_vs_mid_stay_pct         -12.791      drop_evt_mid_stay_lag0      drop_evt_mid_stay_any_lag1_3
        Multi         premium_vs_multi_pct         delta_premium_vs_multi_pct         -11.608         drop_evt_multi_lag0         drop_evt_multi_any_lag1_3
Plane Parking premium_vs_plane_parking_pct delta_premium_vs_plane_parking_pct         -18.355 drop_evt_plane_parking_lag0 drop_evt_plane_parking_any_lag1_3
     Terminal      premium_vs_terminal_pct      delta_premium_vs_terminal_pct         -12.510      drop_evt_terminal_lag0      drop_evt_terminal_any

C:\Users\jamie_douglas\AppData\Local\Temp\ipykernel_6556\3328546122.py:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  diag[f"drop_evt_{comp.lower().replace(' ', '_')}_lag{lag}"] = diag[flag0].shift(lag).fillna(False)
C:\Users\jamie_douglas\AppData\Local\Temp\ipykernel_6556\3328546122.py:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  diag[f"drop_evt_{comp.lower().replace(' ', '_')}_lag{lag}"] = diag[flag0].shift(lag).fillna(False)
C:\Users\jamie_douglas\AppData\Local\Temp\ipykernel_6556\3328546122.py:38: FutureWarning: Downcasting object dtype array

In [58]:
# Compact summary of strongest associations for easy readout
top_lift = (
    event_lift_summary
    .dropna(subset=["lift_ratio"])
    .sort_values("lift_ratio", ascending=False)
    .head(12)
    .copy()
)

drop_day_counts = pd.DataFrame({
    "metric": ["acc_drop_7d_days", "acc_drop_21d_days", "total_summer_days"],
    "value": [int(diag["acc_drop_7d"].sum()), int(diag["acc_drop_21d"].sum()), int(len(diag))],
})

print("Top lift rows:")
print(top_lift.round(3).to_string(index=False))
print("\nDrop day counts:")
print(drop_day_counts.to_string(index=False))

Top lift rows:
   competitor horizon   event_window  event_days  p_acc_drop_given_event  p_acc_drop_given_no_event  lift_ratio
        Multi      7d       same_day           9                   0.444                      0.219       2.030
     Mid Stay     21d       same_day           8                   0.875                      0.450       1.942
        Multi     21d       same_day           9                   0.778                      0.453       1.718
     Mid Stay      7d       same_day           8                   0.375                      0.223       1.683
     Terminal     21d       same_day          10                   0.700                      0.455       1.538
Plane Parking     21d       same_day          10                   0.700                      0.455       1.538
    Long Stay     21d       same_day          10                   0.700                      0.455       1.538
        Multi     21d prior_1_3_days          27                   0.667                 

## 8) Daily/Weekly Graphs with Pricing Event Overlay

> Same core charts, now with pricing-event context on top:
- Red markers = accuracy-drop dates/weeks
- Amber shading = same-day significant FastPark relative price drop vs at least one competitor
- Blue shading = prior 1-3 day significant relative price drop signal

In [60]:
# -----------------------------------------------------------------------------
# OVERLAY HELPERS + ENHANCED DAILY/WEEKLY DUAL-AXIS CHARTS (REVISED CLARITY)
# -----------------------------------------------------------------------------

def _pricing_event_daily_table(lead_col):
    # Keep pricing signals from competitor comparison; accuracy-drop markers are
    # computed directly from the charted accuracy series to avoid confusion.
    out = diag[["Date"]].copy()

    same_day_cols = [m["same_day_flag"] for m in comp_meta]
    prior_cols = [m["lag_any_flag"] for m in comp_meta]
    out["pricing_drop_same_day_any"] = diag[same_day_cols].any(axis=1)
    out["pricing_drop_prior_1_3_any"] = diag[prior_cols].any(axis=1)

    top_comp = []
    top_delta = []
    for _, r in diag.iterrows():
        cands = []
        for m in comp_meta:
            if bool(r.get(m["same_day_flag"], False)):
                cands.append((m["competitor"], r.get(m["delta_col"], np.nan)))
        if cands:
            cands = sorted(cands, key=lambda x: x[1])
            top_comp.append(cands[0][0])
            top_delta.append(cands[0][1])
        else:
            top_comp.append(None)
            top_delta.append(np.nan)

    out["top_comp_same_day"] = top_comp
    out["top_comp_delta_pp"] = top_delta
    return out.sort_values("Date").reset_index(drop=True)

def _pricing_event_weekly_table(event_daily):
    wk = event_daily.copy()
    wk["WeekStart"] = (wk["Date"] - pd.to_timedelta(wk["Date"].dt.weekday, unit="D")).dt.normalize()

    def _top_comp(series):
        s = series.dropna()
        if s.empty:
            return None
        return s.value_counts().idxmax()

    out = (
        wk.groupby("WeekStart", as_index=False)
        .agg(
            pricing_drop_same_day_count=("pricing_drop_same_day_any", "sum"),
            pricing_drop_prior_1_3_count=("pricing_drop_prior_1_3_any", "sum"),
            top_comp_same_day=("top_comp_same_day", _top_comp),
        )
        .rename(columns={"WeekStart": "Date"})
        .sort_values("Date")
    )

    out["pricing_drop_same_day_any"] = out["pricing_drop_same_day_count"] > 0
    out["pricing_drop_prior_1_3_any"] = out["pricing_drop_prior_1_3_count"] > 0
    return out

def _apply_event_overlay(fig, df, event_df, secondary_y=True, row=1, col=1, show_legend=True):
    merge = df[["Date", "AccuracyPct"]].merge(event_df, on="Date", how="left")
    merge["pricing_drop_same_day_any"] = merge["pricing_drop_same_day_any"].fillna(False)
    merge["pricing_drop_prior_1_3_any"] = merge["pricing_drop_prior_1_3_any"].fillna(False)

    # Accuracy-drop markers strictly tied to plotted line (<90%) for clarity.
    merge["accuracy_drop_flag"] = merge["AccuracyPct"] < TARGET_ACCURACY

    # Dynamic shading width to ensure weekly shading is visible.
    if len(merge) > 1:
        cadence_days = merge["Date"].sort_values().diff().dropna().dt.days.median()
        cadence_days = 1 if pd.isna(cadence_days) or cadence_days <= 0 else cadence_days
    else:
        cadence_days = 1
    half_span_hours = float(max(10, cadence_days * 24 * 0.45))

    # Add legend keys once.
    if show_legend:
        fig.add_trace(
            go.Scatter(
                x=[None], y=[None], mode="markers",
                marker=dict(symbol="diamond", size=10, color="#B91C1C", line=dict(color="#7F1D1D", width=1)),
                name="Accuracy drop (<90%)",
            ),
            row=row, col=col, secondary_y=secondary_y,
        )
        fig.add_trace(
            go.Scatter(
                x=[None], y=[None], mode="markers",
                marker=dict(symbol="square", size=10, color="#F59E0B"),
                name="Same-day pricing drop event",
            ),
            row=row, col=col, secondary_y=secondary_y,
        )
        fig.add_trace(
            go.Scatter(
                x=[None], y=[None], mode="markers",
                marker=dict(symbol="square", size=10, color="#60A5FA"),
                name="Prior 1-3 day pricing drop event",
            ),
            row=row, col=col, secondary_y=secondary_y,
        )

    # Background context bands for pricing signals.
    for _, r in merge[merge["pricing_drop_prior_1_3_any"]].iterrows():
        fig.add_vrect(
            x0=r["Date"] - pd.Timedelta(hours=half_span_hours),
            x1=r["Date"] + pd.Timedelta(hours=half_span_hours),
            fillcolor="#60A5FA",
            opacity=0.18,
            line_width=0,
            layer="below",
            row=row,
            col=col,
        )

    for _, r in merge[merge["pricing_drop_same_day_any"]].iterrows():
        fig.add_vrect(
            x0=r["Date"] - pd.Timedelta(hours=half_span_hours),
            x1=r["Date"] + pd.Timedelta(hours=half_span_hours),
            fillcolor="#F59E0B",
            opacity=0.24,
            line_width=0,
            layer="below",
            row=row,
            col=col,
        )

    # Accuracy-drop markers
    mark = merge[merge["accuracy_drop_flag"]].copy()
    fig.add_trace(
        go.Scatter(
            x=mark["Date"],
            y=mark["AccuracyPct"],
            mode="markers",
            name="Accuracy drop (<90%)",
            marker=dict(symbol="diamond", size=10, color="#B91C1C", line=dict(color="#7F1D1D", width=1)),
            hovertemplate=(
                "Date=%{x|%Y-%m-%d}<br>"
                "Accuracy=%{y:.1f}%<br>"
                "Top same-day competitor=%{customdata[0]}<br>"
                "Top premium delta=%{customdata[1]:.2f}pp<extra></extra>"
            ),
            customdata=mark[["top_comp_same_day", "top_comp_delta_pp"]].to_numpy()
            if {"top_comp_same_day", "top_comp_delta_pp"}.issubset(mark.columns)
            else np.column_stack([np.repeat(None, len(mark)), np.repeat(np.nan, len(mark))]),
            showlegend=False,
        ),
        row=row,
        col=col,
        secondary_y=secondary_y,
    )

def plot_dual_axis_total_with_pricing(lead_col, lead_title, grain="daily"):
    ent_daily = _daily_metric_df(entries_accuracy_df, entries_forecast_df, entries_actual_df, lead_col, "Entries")
    exi_daily = _daily_metric_df(exits_accuracy_df, exits_forecast_df, exits_actual_df, lead_col, "Exits")

    combined_daily = pd.DataFrame({
        "Date": ent_daily["Date"].values,
        "Forecast": ent_daily["Forecast"].values + exi_daily["Forecast"].values,
        "Actual": ent_daily["Actual"].values + exi_daily["Actual"].values,
    })
    combined_daily["SignedError"] = combined_daily["Forecast"] - combined_daily["Actual"]
    combined_daily["AccuracyPct"] = np.where(
        combined_daily["Actual"] != 0,
        (1 - np.abs((combined_daily["Actual"] - combined_daily["Forecast"]) / combined_daily["Actual"])) * 100,
        np.nan,
    )

    total_df = _aggregate(combined_daily, grain)

    event_daily = _pricing_event_daily_table(lead_col)
    event_df = _pricing_event_weekly_table(event_daily) if grain == "weekly" else event_daily

    x_label = "Week commencing" if grain == "weekly" else "Arrival date"
    title_grain = "Weekly" if grain == "weekly" else "Daily"

    fig = make_subplots(rows=1, cols=1, specs=[[{"secondary_y": True}]])
    fig.add_trace(
        go.Bar(
            x=total_df["Date"],
            y=total_df["SignedError"],
            name=f"Total {grain} error (Forecast-Actual)",
            marker_color="#4F46E5",
            opacity=0.6,
        ),
        row=1, col=1, secondary_y=False,
    )
    fig.add_trace(
        go.Scatter(
            x=total_df["Date"],
            y=total_df["AccuracyPct"],
            mode="lines+markers",
            name=f"Total {grain} accuracy %",
            line=dict(width=3, color="#0F766E"),
            marker=dict(size=8),
        ),
        row=1, col=1, secondary_y=True,
    )
    fig.add_trace(
        go.Scatter(
            x=total_df["Date"],
            y=np.full(len(total_df), TARGET_ACCURACY),
            mode="lines",
            name="Target accuracy (90%)",
            line=dict(width=2, color="#166534", dash="dash"),
        ),
        row=1, col=1, secondary_y=True,
    )

    _apply_event_overlay(fig, total_df, event_df, secondary_y=True, row=1, col=1, show_legend=True)
    fig.add_hline(y=0, line_width=1.2, line_color="#475569", row=1, col=1, secondary_y=False)

    fig.update_layout(
        **STYLE,
        title=(
            f"Summer 2026 {title_grain} | Total {lead_title}: Signed Error vs Accuracy "
            f"with Pricing Event Overlay"
        ),
        height=580,
        barmode="relative",
        legend=dict(orientation="h", y=1.14, x=0),
    )
    fig.update_yaxes(title_text=f"{title_grain} signed error (Forecast - Actual)", row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text=f"{title_grain} accuracy (%)", range=[40, 100], row=1, col=1, secondary_y=True)
    fig.update_xaxes(title_text=x_label, row=1, col=1)
    fig.show()

def plot_dual_axis_split_with_pricing(lead_col, lead_title, grain="daily"):
    ent_daily = _daily_metric_df(entries_accuracy_df, entries_forecast_df, entries_actual_df, lead_col, "Entries")
    exi_daily = _daily_metric_df(exits_accuracy_df, exits_forecast_df, exits_actual_df, lead_col, "Exits")

    ent = _aggregate(ent_daily, grain)
    exi = _aggregate(exi_daily, grain)

    event_daily = _pricing_event_daily_table(lead_col)
    event_df = _pricing_event_weekly_table(event_daily) if grain == "weekly" else event_daily

    x_label = "Week commencing" if grain == "weekly" else "Arrival date"
    title_grain = "Weekly" if grain == "weekly" else "Daily"

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(f"Entries ({lead_title})", f"Exits ({lead_title})"),
        specs=[[{"secondary_y": True}, {"secondary_y": True}]],
        shared_yaxes=False,
    )

    for i, (df, err_color, acc_color, name) in enumerate([
        (ent, "#2563EB", "#0F766E", "Entries"),
        (exi, "#DC2626", "#C2410C", "Exits"),
    ], start=1):
        fig.add_trace(
            go.Bar(
                x=df["Date"], y=df["SignedError"],
                name=f"{name} {grain} error (Forecast-Actual)",
                marker_color=err_color, opacity=0.55, showlegend=(i == 1),
            ),
            row=1, col=i, secondary_y=False,
        )
        fig.add_trace(
            go.Scatter(
                x=df["Date"], y=df["AccuracyPct"],
                mode="lines+markers",
                name=f"{name} {grain} accuracy %",
                line=dict(width=3, color=acc_color), marker=dict(size=7), showlegend=(i == 1),
            ),
            row=1, col=i, secondary_y=True,
        )
        fig.add_trace(
            go.Scatter(
                x=df["Date"], y=np.full(len(df), TARGET_ACCURACY),
                mode="lines",
                name="Target accuracy (90%)",
                line=dict(width=2, color="#166534", dash="dash"),
                showlegend=(i == 1),
            ),
            row=1, col=i, secondary_y=True,
        )

        _apply_event_overlay(fig, df, event_df, secondary_y=True, row=1, col=i, show_legend=(i == 1))
        fig.add_hline(y=0, line_width=1.2, line_color="#475569", row=1, col=i, secondary_y=False)

    fig.update_layout(
        **STYLE,
        title=(
            f"Summer 2026 {title_grain} | {lead_title}: Signed Error vs Accuracy "
            f"with Pricing Event Overlay"
        ),
        height=620,
        barmode="relative",
        legend=dict(orientation="h", y=1.12, x=0),
    )
    fig.update_yaxes(title_text=f"{title_grain} signed error (Forecast - Actual)", row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text=f"{title_grain} signed error (Forecast - Actual)", row=1, col=2, secondary_y=False)
    fig.update_yaxes(title_text=f"{title_grain} accuracy (%)", range=[40, 100], row=1, col=1, secondary_y=True)
    fig.update_yaxes(title_text=f"{title_grain} accuracy (%)", range=[40, 100], row=1, col=2, secondary_y=True)
    fig.update_xaxes(title_text=x_label, row=1, col=1)
    fig.update_xaxes(title_text=x_label, row=1, col=2)
    fig.show()

# Render enhanced versions (daily + weekly, split + total)
plot_dual_axis_split_with_pricing("3_weeks_out", "3-Week-Out", grain="daily")
plot_dual_axis_split_with_pricing("1_week_out", "1-Week-Out", grain="daily")
plot_dual_axis_total_with_pricing("3_weeks_out", "3-Week-Out", grain="daily")
plot_dual_axis_total_with_pricing("1_week_out", "1-Week-Out", grain="daily")

plot_dual_axis_split_with_pricing("3_weeks_out", "3-Week-Out", grain="weekly")
plot_dual_axis_split_with_pricing("1_week_out", "1-Week-Out", grain="weekly")
plot_dual_axis_total_with_pricing("3_weeks_out", "3-Week-Out", grain="weekly")
plot_dual_axis_total_with_pricing("1_week_out", "1-Week-Out", grain="weekly")

## 9) Reassessment: Week-Out Pricing vs Booking Curve Timing

> Hypothesis: the big misses may be driven more by booking-timing shifts (when people book) than by same-day price level alone.

In [61]:
# -----------------------------------------------------------------------------
# WEEK-OUT BOOKING CURVE + PRICING DRIVERS AROUND ACCURACY DROPS
# -----------------------------------------------------------------------------

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1) Build daily total accuracy/signed error (1W) to identify drop dates
ent_daily = _daily_metric_df(entries_accuracy_df, entries_forecast_df, entries_actual_df, "1_week_out", "Entries")
exi_daily = _daily_metric_df(exits_accuracy_df, exits_forecast_df, exits_actual_df, "1_week_out", "Exits")
total_daily_1w = pd.DataFrame({
    "Date": ent_daily["Date"].values,
    "Forecast": ent_daily["Forecast"].values + exi_daily["Forecast"].values,
    "Actual": ent_daily["Actual"].values + exi_daily["Actual"].values,
})
total_daily_1w["SignedError"] = total_daily_1w["Forecast"] - total_daily_1w["Actual"]
total_daily_1w["AccuracyPct"] = np.where(
    total_daily_1w["Actual"] != 0,
    (1 - np.abs((total_daily_1w["Actual"] - total_daily_1w["Forecast"]) / total_daily_1w["Actual"])) * 100,
    np.nan,
)
total_daily_1w = total_daily_1w.sort_values("Date")

# 2) Build FastPark booking-curve metrics by target entry date
fp = bookings_px[bookings_px["assetName"] == "FastPark"].copy()

fp_curve = (
    fp.groupby("entry_date", as_index=False)
    .agg(
        fp_bookings_total=("bookingId", "nunique"),
        fp_median_lead_days=("lead_days", "median"),
        fp_mean_lead_days=("lead_days", "mean"),
        fp_median_price_day_all=("price_per_day", "median"),
        fp_bookings_0_7=("lead_days", lambda s: s.between(0, 7, inclusive="both").sum()),
        fp_bookings_8_21=("lead_days", lambda s: s.between(8, 21, inclusive="both").sum()),
        fp_bookings_22_120=("lead_days", lambda s: s.between(22, 120, inclusive="both").sum()),
        fp_weekout_ppd=("price_per_day", lambda s: s[fp.loc[s.index, "lead_days"].between(6, 8, inclusive="both")].median()),
    )
    .rename(columns={"entry_date": "Date"})
)
fp_curve["fp_share_0_7"] = fp_curve["fp_bookings_0_7"] / fp_curve["fp_bookings_total"].replace(0, np.nan)
fp_curve["fp_share_8_21"] = fp_curve["fp_bookings_8_21"] / fp_curve["fp_bookings_total"].replace(0, np.nan)

# 3) Competitor week-out price context (individual)
comp_weekout = (
    bookings_px[bookings_px["assetName"].isin(COMPETITORS)]
    .loc[lambda d: d["lead_days"].between(6, 8, inclusive="both")]
    .groupby(["entry_date", "assetName"], as_index=False)["price_per_day"]
    .median()
    .pivot(index="entry_date", columns="assetName", values="price_per_day")
    .reset_index()
    .rename(columns={"entry_date": "Date"})
)

wk = fp_curve.merge(comp_weekout, on="Date", how="left")
for comp in COMPETITORS:
    wk[f"premium_weekout_vs_{comp.lower().replace(' ', '_')}_pct"] = (wk["fp_weekout_ppd"] / wk[comp] - 1) * 100
    wk[f"delta_premium_weekout_vs_{comp.lower().replace(' ', '_')}_pp"] = wk[f"premium_weekout_vs_{comp.lower().replace(' ', '_')}_pct"].diff()

# 4) Merge with forecast outcomes + focus periods
wk_diag = total_daily_1w.merge(wk, on="Date", how="left")
wk_diag = wk_diag[(wk_diag["Date"] >= SUMMER_START) & (wk_diag["Date"] <= SUMMER_END)].copy()
wk_diag["acc_drop_1w"] = wk_diag["AccuracyPct"] < 90
wk_diag["month"] = wk_diag["Date"].dt.month

# month focus: Apr (4), Jun (6), Sep (9)
focus = wk_diag[wk_diag["month"].isin([4, 6, 9])].copy()

# top drop dates in those months
focus_drop_dates = (
    focus.sort_values("AccuracyPct")
    .head(18)
    [[
        "Date", "AccuracyPct", "SignedError", "fp_share_0_7", "fp_share_8_21",
        "fp_median_lead_days", "fp_weekout_ppd", "fp_median_price_day_all",
    ]]
    .copy()
)

# add competitor premium columns for clarity
prem_cols = [f"premium_weekout_vs_{c.lower().replace(' ', '_')}_pct" for c in COMPETITORS]
for c in prem_cols:
    focus_drop_dates[c] = focus.set_index("Date").reindex(focus_drop_dates["Date"])[c].values

# 5) Correlation: timing first, then price
corr_cols = [
    "AccuracyPct", "SignedError",
    "fp_share_0_7", "fp_share_8_21", "fp_median_lead_days", "fp_weekout_ppd",
    "fp_median_price_day_all",
 ] + prem_cols
wk_corr = wk_diag[corr_cols].corr(numeric_only=True)

print("Focus drop dates (Apr/Jun/Sep, lowest 1W accuracy):")
print(focus_drop_dates.round(3).to_string(index=False))

print("\nCorrelation snapshot (1W accuracy/signed error vs booking-curve + week-out price metrics):")
print(wk_corr[["AccuracyPct", "SignedError"]].round(3).to_string())

# 6) Visual reassessment: accuracy + booking timing + week-out premium (top signal competitor)
signal_candidates = []
for comp in COMPETITORS:
    col = f"premium_weekout_vs_{comp.lower().replace(' ', '_')}_pct"
    r = wk_diag[col].corr(wk_diag["AccuracyPct"])
    if pd.notna(r):
        signal_candidates.append((comp, abs(r), r))

if signal_candidates:
    signal_candidates = sorted(signal_candidates, key=lambda x: x[1], reverse=True)
    top_comp = signal_candidates[0][0]
else:
    top_comp = "Long Stay"

top_col = f"premium_weekout_vs_{top_comp.lower().replace(' ', '_')}_pct"

fig_reassess = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06, specs=[[{"secondary_y": False}], [{"secondary_y": False}], [{"secondary_y": True}]])

fig_reassess.add_trace(
    go.Scatter(x=wk_diag["Date"], y=wk_diag["AccuracyPct"], mode="lines+markers", name="1W accuracy %", line=dict(color="#0F766E", width=2.8)),
    row=1, col=1,
 )
fig_reassess.add_trace(
    go.Scatter(x=wk_diag.loc[wk_diag["acc_drop_1w"], "Date"], y=wk_diag.loc[wk_diag["acc_drop_1w"], "AccuracyPct"], mode="markers", name="Accuracy <90", marker=dict(color="#B91C1C", size=9, symbol="diamond")),
    row=1, col=1,
 )
fig_reassess.add_hline(y=90, line_dash="dash", line_color="#166534", row=1, col=1)

fig_reassess.add_trace(
    go.Scatter(x=wk_diag["Date"], y=wk_diag["fp_share_0_7"] * 100, mode="lines", name="FastPark booked <=7 days (%)", line=dict(color="#2563EB", width=2.5)),
    row=2, col=1,
 )
fig_reassess.add_trace(
    go.Scatter(x=wk_diag["Date"], y=wk_diag["fp_median_lead_days"], mode="lines", name="FastPark median lead days", line=dict(color="#7C3AED", width=2), yaxis="y3"),
    row=3, col=1, secondary_y=False,
 )
fig_reassess.add_trace(
    go.Scatter(x=wk_diag["Date"], y=wk_diag[top_col], mode="lines", name=f"Week-out premium vs {top_comp} (%)", line=dict(color="#EA580C", width=2.6)),
    row=3, col=1, secondary_y=True,
 )

for d in wk_diag.loc[wk_diag["acc_drop_1w"], "Date"]:
    fig_reassess.add_vline(x=d, line_color="#FCA5A5", line_width=1, line_dash="dot", row=1, col=1)
    fig_reassess.add_vline(x=d, line_color="#FCA5A5", line_width=1, line_dash="dot", row=2, col=1)
    fig_reassess.add_vline(x=d, line_color="#FCA5A5", line_width=1, line_dash="dot", row=3, col=1)

fig_reassess.update_layout(
    template="plotly_white",
    height=900,
    title=f"Reassessment: 1W Accuracy vs Booking Curve Timing and Week-Out Premium (Top comparator: {top_comp})",
    legend=dict(orientation="h", y=1.04, x=0),
    margin=dict(l=70, r=60, t=80, b=60),
 )
fig_reassess.update_yaxes(title_text="Accuracy (%)", row=1, col=1)
fig_reassess.update_yaxes(title_text="Booked <=7 days (%)", row=2, col=1)
fig_reassess.update_yaxes(title_text="Median lead days", row=3, col=1, secondary_y=False)
fig_reassess.update_yaxes(title_text="Week-out premium (%)", row=3, col=1, secondary_y=True)
fig_reassess.update_xaxes(title_text="Arrival date", row=3, col=1)

fig_reassess.show()

Focus drop dates (Apr/Jun/Sep, lowest 1W accuracy):
      Date  AccuracyPct  SignedError  fp_share_0_7  fp_share_8_21  fp_median_lead_days  fp_weekout_ppd  fp_median_price_day_all  premium_weekout_vs_long_stay_pct  premium_weekout_vs_mid_stay_pct  premium_weekout_vs_multi_pct  premium_weekout_vs_plane_parking_pct  premium_weekout_vs_terminal_pct
2026-09-02       57.859        437.0         0.197          0.378                 18.0          16.594                   13.993                            16.712                          -13.549                       -48.826                                61.346                          -38.519
2026-09-01       59.721        433.0         0.221          0.391                 16.0          16.491                   14.338                            10.413                          -11.643                       -47.081                                60.383                          -32.647
2026-09-05       70.091        396.0         0.232          

In [62]:
# Compact readout for reassessment conclusions
summary_cols = [
    "Date", "AccuracyPct", "SignedError", "fp_share_0_7", "fp_median_lead_days",
    "fp_weekout_ppd"
] + [f"premium_weekout_vs_{c.lower().replace(' ', '_')}_pct" for c in COMPETITORS]

worst_10 = wk_diag.sort_values("AccuracyPct").head(10)[summary_cols].copy()

corr_focus = wk_corr.loc[[
    "fp_share_0_7", "fp_share_8_21", "fp_median_lead_days", "fp_weekout_ppd", "fp_median_price_day_all"
] + [f"premium_weekout_vs_{c.lower().replace(' ', '_')}_pct" for c in COMPETITORS], ["AccuracyPct", "SignedError"]]

print("Worst 10 days by 1W accuracy:")
print(worst_10.round(3).to_string(index=False))

print("\nCorrelation focus (booking curve + week-out pricing vs outcomes):")
print(corr_focus.round(3).to_string())

Worst 10 days by 1W accuracy:
      Date  AccuracyPct  SignedError  fp_share_0_7  fp_median_lead_days  fp_weekout_ppd  premium_weekout_vs_long_stay_pct  premium_weekout_vs_mid_stay_pct  premium_weekout_vs_multi_pct  premium_weekout_vs_plane_parking_pct  premium_weekout_vs_terminal_pct
2026-09-02       57.859        437.0         0.197                 18.0          16.594                            16.712                          -13.549                       -48.826                                61.346                          -38.519
2026-09-01       59.721        433.0         0.221                 16.0          16.491                            10.413                          -11.643                       -47.081                                60.383                          -32.647
2026-08-31       68.957        375.0         0.303                 14.0          14.999                             8.001                          -21.036                       -45.890                  

## 10) Driver Diagnostics: Why Booking Behaviour Shifted

> Goal: move from correlation to likely drivers by comparing major drop dates against matched baseline days (same weekday, recent history).

In [63]:
# -----------------------------------------------------------------------------
# DRIVER DIAGNOSTICS: MATCHED-BASELINE EXPLANATION FOR ACCURACY DROP DATES
# -----------------------------------------------------------------------------

# Build day-level demand/price behavior features
day_asset = (
    bookings_px.groupby(["entry_date", "assetName"], as_index=False)
    .agg(
        bookings=("bookingId", "nunique"),
        median_lead_days=("lead_days", "median"),
        share_0_7=("lead_days", lambda s: s.between(0, 7, inclusive="both").mean()),
        weekout_ppd=("price_per_day", lambda s: s[bookings_px.loc[s.index, "lead_days"].between(6, 8, inclusive="both")].median()),
        median_ppd=("price_per_day", "median"),
    )
    .rename(columns={"entry_date": "Date"})
)

wide_bookings = day_asset.pivot(index="Date", columns="assetName", values="bookings")
wide_weekout_ppd = day_asset.pivot(index="Date", columns="assetName", values="weekout_ppd")
wide_median_ppd = day_asset.pivot(index="Date", columns="assetName", values="median_ppd")
wide_share_07 = day_asset.pivot(index="Date", columns="assetName", values="share_0_7")
wide_lead = day_asset.pivot(index="Date", columns="assetName", values="median_lead_days")

drv = pd.DataFrame(index=sorted(set(wide_bookings.index).intersection(set(total_daily_1w["Date"]))))
drv.index.name = "Date"
drv = drv.reset_index()

# Forecast outcome context
drv = drv.merge(total_daily_1w[["Date", "AccuracyPct", "SignedError"]], on="Date", how="left")
drv["acc_drop_1w"] = drv["AccuracyPct"] < 90
drv["month"] = drv["Date"].dt.month
drv["weekday"] = drv["Date"].dt.day_name()

# FastPark behavior
drv["fp_bookings"] = drv["Date"].map(wide_bookings.get("FastPark", pd.Series(dtype=float)))
drv["fp_share_0_7"] = drv["Date"].map(wide_share_07.get("FastPark", pd.Series(dtype=float)))
drv["fp_median_lead_days"] = drv["Date"].map(wide_lead.get("FastPark", pd.Series(dtype=float)))
drv["fp_weekout_ppd"] = drv["Date"].map(wide_weekout_ppd.get("FastPark", pd.Series(dtype=float)))
drv["fp_median_ppd"] = drv["Date"].map(wide_median_ppd.get("FastPark", pd.Series(dtype=float)))

# Market context
drv["market_bookings"] = wide_bookings.sum(axis=1).reindex(drv["Date"]).values
drv["fp_market_share"] = drv["fp_bookings"] / drv["market_bookings"].replace(0, np.nan)
drv["market_share_0_7"] = wide_share_07.mean(axis=1).reindex(drv["Date"]).values

# Competitor relative pricing features (week-out)
comp_weekout_cols = [c for c in COMPETITORS if c in wide_weekout_ppd.columns]
if comp_weekout_cols:
    comp_weekout_med = wide_weekout_ppd[comp_weekout_cols].median(axis=1)
    comp_weekout_std = wide_weekout_ppd[comp_weekout_cols].std(axis=1)
    comp_weekout_min = wide_weekout_ppd[comp_weekout_cols].min(axis=1)
else:
    comp_weekout_med = pd.Series(index=wide_weekout_ppd.index, dtype=float)
    comp_weekout_std = pd.Series(index=wide_weekout_ppd.index, dtype=float)
    comp_weekout_min = pd.Series(index=wide_weekout_ppd.index, dtype=float)

drv["comp_weekout_median_ppd"] = drv["Date"].map(comp_weekout_med)
drv["comp_weekout_std_ppd"] = drv["Date"].map(comp_weekout_std)
drv["comp_weekout_min_ppd"] = drv["Date"].map(comp_weekout_min)
drv["fp_vs_comp_weekout_pct"] = (drv["fp_weekout_ppd"] / drv["comp_weekout_median_ppd"] - 1) * 100
drv["fp_vs_cheapest_comp_weekout_pct"] = (drv["fp_weekout_ppd"] / drv["comp_weekout_min_ppd"] - 1) * 100

# Day-to-day changes (possible shock indicators)
for c in [
    "fp_share_0_7", "fp_median_lead_days", "fp_market_share", "fp_weekout_ppd",
    "fp_vs_comp_weekout_pct", "market_bookings", "market_share_0_7",
    "comp_weekout_std_ppd"
]:
    drv[f"d1_{c}"] = drv[c].diff()

# Month focus with major drops
focus = drv[drv["month"].isin([4, 6, 9])].copy()
major_drop_dates = focus.nsmallest(15, "AccuracyPct")["Date"].tolist()

# Matched baseline: same weekday from previous 4 weeks
def matched_baseline(df, target_date, lookback_days=56):
    wd = target_date.day_name()
    hist = df[(df["Date"] < target_date) & (df["Date"] >= target_date - pd.Timedelta(days=lookback_days)) & (df["weekday"] == wd)].copy()
    return hist.tail(4)

feature_cols = [
    "fp_share_0_7", "fp_median_lead_days", "fp_market_share", "fp_weekout_ppd",
    "fp_vs_comp_weekout_pct", "fp_vs_cheapest_comp_weekout_pct",
"market_bookings", "market_share_0_7", "comp_weekout_std_ppd",
"d1_fp_share_0_7", "d1_fp_median_lead_days", "d1_fp_market_share",
"d1_fp_weekout_ppd", "d1_fp_vs_comp_weekout_pct", "d1_market_bookings",
"d1_market_share_0_7", "d1_comp_weekout_std_ppd"
]

driver_rows = []
for dt in major_drop_dates:
    row = drv[drv["Date"] == dt].iloc[0]
    base = matched_baseline(drv, dt)
    if len(base) < 2:
        continue

    for f in feature_cols:
        cur = row.get(f, np.nan)
        mu = base[f].mean()
        sd = base[f].std()
        z = (cur - mu) / sd if pd.notna(sd) and sd > 0 else np.nan
        driver_rows.append({
            "Date": dt,
            "feature": f,
            "current": cur,
            "baseline_mean": mu,
            "delta": cur - mu if pd.notna(cur) and pd.notna(mu) else np.nan,
            "z_score": z,
            "accuracy_pct": row["AccuracyPct"],
            "signed_error": row["SignedError"],
        })

driver_diag = pd.DataFrame(driver_rows)

# Per-date top abnormal signals (by absolute z-score)
top_driver_by_date = (
    driver_diag.dropna(subset=["z_score"])
    .assign(abs_z=lambda d: d["z_score"].abs())
    .sort_values(["Date", "abs_z"], ascending=[True, False])
    .groupby("Date", as_index=False)
    .head(5)
[["Date", "accuracy_pct", "signed_error", "feature", "current", "baseline_mean", "delta", "z_score"]]
)

# Aggregate feature importance proxy across drop dates
feature_strength = (
    driver_diag.dropna(subset=["z_score"])
    .assign(abs_z=lambda d: d["z_score"].abs())
    .groupby("feature", as_index=False)
    .agg(
        mean_abs_z=("abs_z", "mean"),
        median_abs_z=("abs_z", "median"),
        pos_count=("z_score", lambda s: (s > 0).sum()),
        neg_count=("z_score", lambda s: (s < 0).sum()),
        n=("z_score", "count")
    )
    .sort_values("mean_abs_z", ascending=False)
 )

print("Major drop dates used (Apr/Jun/Sep focus):")
print(pd.Series(major_drop_dates).dt.strftime("%Y-%m-%d").to_list())

print("\nTop 5 abnormal driver signals per drop date (vs matched baseline):")
print(top_driver_by_date.round(3).to_string(index=False))

print("\nMost consistently abnormal features across drop dates:")
print(feature_strength.head(15).round(3).to_string(index=False))

Major drop dates used (Apr/Jun/Sep focus):
['2026-09-02', '2026-09-01', '2026-09-05', '2026-09-03', '2026-09-19', '2026-09-08', '2026-09-16', '2026-09-04', '2026-04-01', '2026-04-15', '2026-09-06', '2026-09-17', '2026-04-10', '2026-04-08', '2026-09-07']

Top 5 abnormal driver signals per drop date (vs matched baseline):
      Date  accuracy_pct  signed_error                         feature  current  baseline_mean    delta  z_score
2026-04-15        80.490        -215.0             d1_market_share_0_7   -0.011         -0.030    0.019   21.749
2026-04-15        80.490        -215.0                  fp_weekout_ppd   17.377         15.142    2.235   19.536
2026-04-15        80.490        -215.0                 market_bookings 1875.000       1674.000  201.000   10.933
2026-04-15        80.490        -215.0                    fp_share_0_7    0.492          0.213    0.279    3.073
2026-04-15        80.490        -215.0             fp_median_lead_days    8.000         29.500  -21.500   -2.027


In [64]:
# Plain-English driver summary for slide drafting
top_features = feature_strength.head(8).copy()

print("Likely driver themes (based on repeated abnormality vs matched baseline):")
for i, r in top_features.iterrows():
    f = r["feature"]
    direction = "mixed"
    if r["pos_count"] > r["neg_count"]:
        direction = "mostly higher than baseline"
    elif r["neg_count"] > r["pos_count"]:
        direction = "mostly lower than baseline"
    print(f"- {f}: mean |z|={r['mean_abs_z']:.2f}, {direction} (n={int(r['n'])})")

print("\nInterpretation hint:")
print("Higher |z| means the feature moved unusually vs same-weekday recent history on drop dates.")

Likely driver themes (based on repeated abnormality vs matched baseline):
- fp_vs_cheapest_comp_weekout_pct: mean |z|=4.15, mixed (n=12)
- fp_weekout_ppd: mean |z|=3.74, mostly higher than baseline (n=12)
- d1_market_share_0_7: mean |z|=2.71, mostly higher than baseline (n=12)
- market_bookings: mean |z|=2.60, mostly higher than baseline (n=12)
- d1_fp_median_lead_days: mean |z|=2.51, mixed (n=12)
- d1_fp_share_0_7: mean |z|=2.14, mostly lower than baseline (n=12)
- d1_market_bookings: mean |z|=1.80, mostly higher than baseline (n=12)
- fp_median_lead_days: mean |z|=1.65, mostly higher than baseline (n=12)

Interpretation hint:
Higher |z| means the feature moved unusually vs same-weekday recent history on drop dates.


## 11) Driver Diagnostics: 3-Week-Out Accuracy (Same Framework)

> Replicate the same matched-baseline driver analysis for 3-week-out accuracy so we can compare 1W vs 3W drivers directly.

In [65]:
# -----------------------------------------------------------------------------
# DRIVER DIAGNOSTICS (3W): MATCHED-BASELINE EXPLANATION FOR DROP DATES
# -----------------------------------------------------------------------------

# Build 3-week-out daily total accuracy/signed error
ent_daily_3w = _daily_metric_df(entries_accuracy_df, entries_forecast_df, entries_actual_df, "3_weeks_out", "Entries")
exi_daily_3w = _daily_metric_df(exits_accuracy_df, exits_forecast_df, exits_actual_df, "3_weeks_out", "Exits")
total_daily_3w = pd.DataFrame({
    "Date": ent_daily_3w["Date"].values,
    "Forecast": ent_daily_3w["Forecast"].values + exi_daily_3w["Forecast"].values,
    "Actual": ent_daily_3w["Actual"].values + exi_daily_3w["Actual"].values,
})
total_daily_3w["SignedError"] = total_daily_3w["Forecast"] - total_daily_3w["Actual"]
total_daily_3w["AccuracyPct"] = np.where(
    total_daily_3w["Actual"] != 0,
    (1 - np.abs((total_daily_3w["Actual"] - total_daily_3w["Forecast"]) / total_daily_3w["Actual"])) * 100,
    np.nan,
 )
total_daily_3w = total_daily_3w.sort_values("Date")

# Rebuild day-level booking/price matrices if needed
if "wide_bookings" not in globals():
    day_asset = (
        bookings_px.groupby(["entry_date", "assetName"], as_index=False)
        .agg(
            bookings=("bookingId", "nunique"),
            median_lead_days=("lead_days", "median"),
            share_0_7=("lead_days", lambda s: s.between(0, 7, inclusive="both").mean()),
            weekout_ppd=("price_per_day", lambda s: s[bookings_px.loc[s.index, "lead_days"].between(6, 8, inclusive="both")].median()),
            median_ppd=("price_per_day", "median"),
        )
        .rename(columns={"entry_date": "Date"})
    )
    wide_bookings = day_asset.pivot(index="Date", columns="assetName", values="bookings")
    wide_weekout_ppd = day_asset.pivot(index="Date", columns="assetName", values="weekout_ppd")
    wide_median_ppd = day_asset.pivot(index="Date", columns="assetName", values="median_ppd")
    wide_share_07 = day_asset.pivot(index="Date", columns="assetName", values="share_0_7")
    wide_lead = day_asset.pivot(index="Date", columns="assetName", values="median_lead_days")

drv3 = pd.DataFrame(index=sorted(set(wide_bookings.index).intersection(set(total_daily_3w["Date"]))))
drv3.index.name = "Date"
drv3 = drv3.reset_index()

drv3 = drv3.merge(total_daily_3w[["Date", "AccuracyPct", "SignedError"]], on="Date", how="left")
drv3["acc_drop_3w"] = drv3["AccuracyPct"] < 90
drv3["month"] = drv3["Date"].dt.month
drv3["weekday"] = drv3["Date"].dt.day_name()

drv3["fp_bookings"] = drv3["Date"].map(wide_bookings.get("FastPark", pd.Series(dtype=float)))
drv3["fp_share_0_7"] = drv3["Date"].map(wide_share_07.get("FastPark", pd.Series(dtype=float)))
drv3["fp_median_lead_days"] = drv3["Date"].map(wide_lead.get("FastPark", pd.Series(dtype=float)))
drv3["fp_weekout_ppd"] = drv3["Date"].map(wide_weekout_ppd.get("FastPark", pd.Series(dtype=float)))
drv3["fp_median_ppd"] = drv3["Date"].map(wide_median_ppd.get("FastPark", pd.Series(dtype=float)))

drv3["market_bookings"] = wide_bookings.sum(axis=1).reindex(drv3["Date"]).values
drv3["fp_market_share"] = drv3["fp_bookings"] / drv3["market_bookings"].replace(0, np.nan)
drv3["market_share_0_7"] = wide_share_07.mean(axis=1).reindex(drv3["Date"]).values

comp_weekout_cols = [c for c in COMPETITORS if c in wide_weekout_ppd.columns]
if comp_weekout_cols:
    comp_weekout_med = wide_weekout_ppd[comp_weekout_cols].median(axis=1)
    comp_weekout_std = wide_weekout_ppd[comp_weekout_cols].std(axis=1)
    comp_weekout_min = wide_weekout_ppd[comp_weekout_cols].min(axis=1)
else:
    comp_weekout_med = pd.Series(index=wide_weekout_ppd.index, dtype=float)
    comp_weekout_std = pd.Series(index=wide_weekout_ppd.index, dtype=float)
    comp_weekout_min = pd.Series(index=wide_weekout_ppd.index, dtype=float)

drv3["comp_weekout_median_ppd"] = drv3["Date"].map(comp_weekout_med)
drv3["comp_weekout_std_ppd"] = drv3["Date"].map(comp_weekout_std)
drv3["comp_weekout_min_ppd"] = drv3["Date"].map(comp_weekout_min)
drv3["fp_vs_comp_weekout_pct"] = (drv3["fp_weekout_ppd"] / drv3["comp_weekout_median_ppd"] - 1) * 100
drv3["fp_vs_cheapest_comp_weekout_pct"] = (drv3["fp_weekout_ppd"] / drv3["comp_weekout_min_ppd"] - 1) * 100

for c in [
    "fp_share_0_7", "fp_median_lead_days", "fp_market_share", "fp_weekout_ppd",
    "fp_vs_comp_weekout_pct", "market_bookings", "market_share_0_7", "comp_weekout_std_ppd",
 ]:
    drv3[f"d1_{c}"] = drv3[c].diff()

focus3 = drv3[drv3["month"].isin([4, 6, 9])].copy()
major_drop_dates_3w = focus3.nsmallest(15, "AccuracyPct")["Date"].tolist()

if "matched_baseline" not in globals():
    def matched_baseline(df, target_date, lookback_days=56):
        wd = target_date.day_name()
        hist = df[(df["Date"] < target_date) & (df["Date"] >= target_date - pd.Timedelta(days=lookback_days)) & (df["weekday"] == wd)].copy()
        return hist.tail(4)

feature_cols_3w = [
    "fp_share_0_7", "fp_median_lead_days", "fp_market_share", "fp_weekout_ppd",
    "fp_vs_comp_weekout_pct", "fp_vs_cheapest_comp_weekout_pct",
    "market_bookings", "market_share_0_7", "comp_weekout_std_ppd",
    "d1_fp_share_0_7", "d1_fp_median_lead_days", "d1_fp_market_share",
    "d1_fp_weekout_ppd", "d1_fp_vs_comp_weekout_pct", "d1_market_bookings",
    "d1_market_share_0_7", "d1_comp_weekout_std_ppd",
 ]

driver_rows_3w = []
for dt in major_drop_dates_3w:
    row = drv3[drv3["Date"] == dt].iloc[0]
    base = matched_baseline(drv3, dt)
    if len(base) < 2:
        continue

    for f in feature_cols_3w:
        cur = row.get(f, np.nan)
        mu = base[f].mean()
        sd = base[f].std()
        z = (cur - mu) / sd if pd.notna(sd) and sd > 0 else np.nan
        driver_rows_3w.append({
            "Date": dt,
            "feature": f,
            "current": cur,
            "baseline_mean": mu,
            "delta": cur - mu if pd.notna(cur) and pd.notna(mu) else np.nan,
            "z_score": z,
            "accuracy_pct": row["AccuracyPct"],
            "signed_error": row["SignedError"],
        })

driver_diag_3w = pd.DataFrame(driver_rows_3w)

top_driver_by_date_3w = (
    driver_diag_3w.dropna(subset=["z_score"])
    .assign(abs_z=lambda d: d["z_score"].abs())
    .sort_values(["Date", "abs_z"], ascending=[True, False])
    .groupby("Date", as_index=False)
    .head(5)
    [["Date", "accuracy_pct", "signed_error", "feature", "current", "baseline_mean", "delta", "z_score"]]
)

feature_strength_3w = (
    driver_diag_3w.dropna(subset=["z_score"])
    .assign(abs_z=lambda d: d["z_score"].abs())
    .groupby("feature", as_index=False)
    .agg(
        mean_abs_z=("abs_z", "mean"),
        median_abs_z=("abs_z", "median"),
        pos_count=("z_score", lambda s: (s > 0).sum()),
        neg_count=("z_score", lambda s: (s < 0).sum()),
        n=("z_score", "count"),
    )
    .sort_values("mean_abs_z", ascending=False)
 )

print("3W major drop dates used (Apr/Jun/Sep focus):")
print(pd.Series(major_drop_dates_3w).dt.strftime("%Y-%m-%d").to_list())

print("\n3W top 5 abnormal driver signals per drop date (vs matched baseline):")
print(top_driver_by_date_3w.round(3).to_string(index=False))

print("\n3W most consistently abnormal features across drop dates:")
print(feature_strength_3w.head(15).round(3).to_string(index=False))

3W major drop dates used (Apr/Jun/Sep focus):
['2026-06-19', '2026-04-21', '2026-04-22', '2026-04-25', '2026-04-08', '2026-04-07', '2026-06-21', '2026-04-28', '2026-06-20', '2026-04-30', '2026-04-14', '2026-04-29', '2026-04-15', '2026-06-18', '2026-04-06']

3W top 5 abnormal driver signals per drop date (vs matched baseline):
      Date  accuracy_pct  signed_error                         feature  current  baseline_mean    delta  z_score
2026-04-14        67.225         313.0                    fp_share_0_7    0.453          0.214    0.238    8.320
2026-04-14        67.225         313.0                 fp_market_share    0.264          0.226    0.038    3.867
2026-04-14        67.225         313.0          fp_vs_comp_weekout_pct    2.974         -6.697    9.671    2.947
2026-04-14        67.225         313.0                  fp_weekout_ppd   17.332         14.968    2.364    2.842
2026-04-14        67.225         313.0             fp_median_lead_days    9.500         26.500  -17.000   -

In [66]:
# Side-by-side 1W vs 3W comparison (same features)
cmp = feature_strength[["feature", "mean_abs_z", "median_abs_z"]].rename(
    columns={"mean_abs_z": "mean_abs_z_1w", "median_abs_z": "median_abs_z_1w"}
).merge(
    feature_strength_3w[["feature", "mean_abs_z", "median_abs_z"]].rename(
        columns={"mean_abs_z": "mean_abs_z_3w", "median_abs_z": "median_abs_z_3w"}
    ),
    on="feature", how="outer"
 )

cmp["delta_mean_abs_z_3w_minus_1w"] = cmp["mean_abs_z_3w"] - cmp["mean_abs_z_1w"]
cmp = cmp.sort_values("delta_mean_abs_z_3w_minus_1w", ascending=False)

print("Feature abnormality comparison: 3W vs 1W")
print(cmp.round(3).to_string(index=False))

print("\nTop drivers for 3W (by mean |z|):")
for _, r in feature_strength_3w.head(8).iterrows():
    direction = "mixed"
    if r["pos_count"] > r["neg_count"]:
        direction = "mostly higher than baseline"
    elif r["neg_count"] > r["pos_count"]:
        direction = "mostly lower than baseline"
    print(f"- {r['feature']}: mean |z|={r['mean_abs_z']:.2f}, {direction} (n={int(r['n'])})")

Feature abnormality comparison: 3W vs 1W
                        feature  mean_abs_z_1w  median_abs_z_1w  mean_abs_z_3w  median_abs_z_3w  delta_mean_abs_z_3w_minus_1w
         fp_vs_comp_weekout_pct          1.143            1.007          3.247            1.816                         2.105
                   fp_share_0_7          1.079            0.721          2.434            2.473                         1.356
        d1_comp_weekout_std_ppd          0.895            0.733          1.813            1.301                         0.919
                fp_market_share          1.040            0.368          1.726            1.736                         0.686
           comp_weekout_std_ppd          1.010            0.593          1.662            1.352                         0.652
      d1_fp_vs_comp_weekout_pct          1.175            0.785          1.673            1.537                         0.498
              d1_fp_weekout_ppd          1.564            1.302          1.96